# ICT4403 Assignment 2 — Autoregressive Next-Word Text Generation

**Models:** Simple RNN, Bidirectional RNN, LSTM, GRU  
**Dataset:** Sherlock Holmes Complete Collection (attached through Kaggle Inputs)  
**Environment:** Kaggle Notebook, PyTorch, one GPU  
**Notebook file:** `fgai_assignment2_text_generation.ipynb`

This notebook implements the workflow in `PLAN.md` and `IMPLEMENTATION_BLUEPRINT.md`. It does not download data or assume internet access. Before running, attach the dataset in **Add Input → Datasets**. Recommended input: [Sherlock Holmes Complete Collection](https://www.kaggle.com/datasets/bengiefigueroa/sherlock-holmes-complete-collection).

## Run instructions

1. In Kaggle, select **GPU** under Notebook Settings → Accelerator (P100/T4 or equivalent).
2. Add the dataset above as an input. Keep internet disabled; this notebook does not fetch data.
3. Run **Save Version → Save and Run All** for a reproducible committed run after testing interactively.
4. Outputs are written under `/kaggle/working/` in `results/`, `plots/`, `samples/`, `checkpoints/`, and `data/`.
5. Check the dataset file manifest and license/source text against the actual Kaggle dataset page before submitting. Do not commit raw text or model checkpoints to GitHub.

The notebook intentionally fails early if a final run has no GPU. For CPU-only debugging, set the environment variable `ALLOW_CPU_DEBUG=1` before executing Section 0; do not use that setting for the final reported training run.

**Important methodology:** examples predict one target token from the preceding 30-token context. The target token is never included in its own input. The bidirectional RNN only reads the observed context window in both directions; it never receives future target tokens.

## Section 0 — Setup, configuration, reproducibility

In [1]:
# This must be the first code cell: configure deterministic cuBLAS before importing torch.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys, re, json, math, time, random, hashlib, inspect, unicodedata, difflib, gc, warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path
from collections import Counter
from typing import Iterable, Sequence, Iterator, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Kaggle default is final-run mode (GPU required). Set ALLOW_CPU_DEBUG=1 only for local/small smoke tests.
ALLOW_CPU_DEBUG = os.environ.get("ALLOW_CPU_DEBUG", "0") == "1"
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(0)
else:
    if not ALLOW_CPU_DEBUG:
        raise RuntimeError("No CUDA GPU found. In Kaggle Settings select a GPU accelerator. For debug only, set ALLOW_CPU_DEBUG=1 before Section 0.")
    device = torch.device("cpu")
    gpu_name = "CPU_DEBUG"

# Determinism is best effort: PyTorch/cuDNN may still have version/platform-specific nondeterminism.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except TypeError:  # Compatibility with older preinstalled torch builds.
    torch.use_deterministic_algorithms(True)

INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
DIRS = {
    "results": OUT / "results",
    "plots": OUT / "plots",
    "samples": OUT / "samples",
    "checkpoints": OUT / "checkpoints",
    "data": OUT / "data",
}
for _p in DIRS.values():
    _p.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ("simple_rnn", "birnn", "lstm", "gru")
MODEL_LABELS = {
    "simple_rnn": "Simple RNN", "birnn": "Bidirectional RNN",
    "lstm": "LSTM", "gru": "GRU", "birnn_matched": "BiRNN matched"
}
MODEL_COLOURS = {
    "simple_rnn": "#1f77b4", "birnn": "#ff7f0e",
    "lstm": "#2ca02c", "gru": "#d62728", "birnn_matched": "#9467bd"
}
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)*|[.,!?;:\"-]")
WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)*")

@dataclass(frozen=True)
class DataCfg:
    L: int = 30
    max_vocab: int = 10_000
    min_freq: int = 3
    split_targets: tuple = (0.80, 0.10, 0.10)
    split_bands: tuple = ((0.78, 0.82), (0.08, 0.12), (0.08, 0.12))
    split_seed_start: int = 42
    split_max_tries: int = 200
    overlap_n: int = 12
    overlap_split_max: float = 0.01
    overlap_work_max: float = 0.30
    min_works: int = 20
    exclude_file_names: tuple = ("cano.txt",)
    train_stride: int = 1
    eval_stride: int = 1

@dataclass(frozen=True)
class ModelCfg:
    embed_dim: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    dropout: float = 0.3

@dataclass(frozen=True)
class TrainCfg:
    lr: float = 0.001
    weight_decay: float = 0.0
    clip_norm: float = 1.0
    batch_size: int = 256
    eval_batch_size: int = 1024
    max_epochs: int = 20
    sched_factor: float = 0.5
    sched_patience: int = 1
    sched_min_lr: float = 0.00001
    es_patience: int = 3
    es_min_delta: float = 0.001
    subsample_n: int = 50_000
    subsample_seed: int = 12345
    topk: int = 5

@dataclass(frozen=True)
class GenCfg:
    temperatures: tuple = (0.5, 1.0, 1.5)
    include_greedy: bool = True
    n_new: int = 50
    samples_per_cell: int = 5
    rng_base: int = 100_000
    seed_texts: tuple = (
        ("S1", "it was a cold foggy morning in baker street and"),
        ("S2", '" my dear watson , " said holmes , "'),
        ("S3", "from the mud upon your boots i can see that you"),
        ("S4", "the inspector knelt beside the body and"),
        ("S5", "a lady in a thick veil was shown into the room and"),
        ("S6", "i have seldom seen my friend so moved as he was when"),
    )
    seed_replacements: tuple = ()

@dataclass(frozen=True)
class RunCfg:
    model_seed: int = 42
    extra_seeds: tuple = (43, 44)
    time_budget_h: float = 6.0
    extra_seed_gate_h: float = 2.0
    ablation_margin: float = 1.5
    gen_reserve_h: float = 0.15
    rating_shuffle_seed: int = 4242

@dataclass(frozen=True)
class Config:
    data: DataCfg = DataCfg()
    model: ModelCfg = ModelCfg()
    train: TrainCfg = TrainCfg()
    gen: GenCfg = GenCfg()
    run: RunCfg = RunCfg()

CFG = Config()
RUN = {
    "cfg_changes": [], "guard_log": [], "warnings": [], "seeds_run": [],
    "optional_ablation_trained": False, "extra_seeds_decision": "pending",
    "small_corpus": False, "removed_near_duplicates": [],
    "started_unix": time.time(), "gpu_name": gpu_name,
    "torch_version": torch.__version__, "numpy_version": np.__version__, "pandas_version": pd.__version__,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _jsonable(obj):
    if hasattr(obj, "__dataclass_fields__"):
        return asdict(obj)
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (Path,)): return str(obj)
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, (np.ndarray,)): return obj.tolist()
    if isinstance(obj, torch.Tensor): return obj.detach().cpu().tolist()
    if isinstance(obj, (set,)): return sorted(_jsonable(x) for x in obj)
    return obj

def write_json(obj, path: Path) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(_jsonable(obj), indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")

def write_csv(df: pd.DataFrame, path: Path) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def write_md_table(df: pd.DataFrame, path: Path, float_fmt: Optional[dict] = None) -> None:
    float_fmt = float_fmt or {}
    out = []
    cols = list(df.columns)
    out.append("| " + " | ".join(str(c).replace("|", "\\|") for c in cols) + " |")
    out.append("| " + " | ".join(["---"] * len(cols)) + " |")
    for _, row in df.iterrows():
        vals=[]
        for c in cols:
            v=row[c]
            if pd.isna(v): s=""
            elif isinstance(v, (float, np.floating)):
                s=("{:." + str(float_fmt.get(c, 4)) + "f}").format(float(v))
            else: s=str(v)
            vals.append(s.replace("|", "\\|").replace("\n", " "))
        out.append("| " + " | ".join(vals) + " |")
    Path(path).write_text("\n".join(out) + "\n", encoding="utf-8")

def shared_digest(cfg: Config) -> str:
    blob = json.dumps(asdict(cfg), sort_keys=True, separators=(",", ":"), default=list).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()

def global_flags_snapshot():
    return {
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
        "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    }

set_seed(CFG.run.model_seed)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(device)

ENV_TEXT = (
    f"python={sys.version}\nplatform={sys.platform}\ntorch={torch.__version__}\n"
    f"torch_cuda={torch.version.cuda}\ncudnn={torch.backends.cudnn.version()}\n"
    f"numpy={np.__version__}\npandas={pd.__version__}\nmatplotlib={plt.matplotlib.__version__}\n"
    f"device={device}\ngpu_name={gpu_name}\nflags={global_flags_snapshot()}\n"
    "Reproducibility: best effort on the same hardware/software; GPU RNN kernels can vary across versions.\n"
)
(DIRS["results"] / "environment.txt").write_text(ENV_TEXT, encoding="utf-8")
write_json({"config": CFG, "device": str(device), "gpu_name": gpu_name,
            "flags": global_flags_snapshot(), "shared_digest": shared_digest(CFG)}, DIRS["results"] / "run_config.json")
print("Device:", device, "|", gpu_name)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
print("Outputs:", OUT.resolve())

Device: cuda:0 | Tesla T4
PyTorch: 2.11.0+cu128 | CUDA: 12.8 | cuDNN: 91900
Outputs: /kaggle/working


## Section 1 — Discover dataset files (no network access)

In [2]:
assert INPUT_ROOT.exists(), f"Input root {INPUT_ROOT} does not exist. Run this notebook in Kaggle with the Sherlock dataset attached."
all_txt = sorted(p for p in INPUT_ROOT.rglob("*.txt") if p.is_file())
excluded = [p for p in all_txt if p.name.lower() in set(CFG.data.exclude_file_names)]
txt_files = [p for p in all_txt if p.name.lower() not in set(CFG.data.exclude_file_names)]

# Fail early if the expected corpus was not attached, rather than silently training on unrelated data.
print(f"Discovered {len(all_txt)} .txt files under {INPUT_ROOT}; excluded duplicate-container candidates: {len(excluded)}")
for p in all_txt[:80]:
    print(" -", p.relative_to(INPUT_ROOT))
if len(txt_files) < CFG.data.min_works:
    raise RuntimeError(
        f"Only {len(txt_files)} text files found (need at least {CFG.data.min_works}). "
        "Attach the Sherlock Holmes Complete Collection dataset or check its input path. "
        "If using the fallback dataset, make sure the duplicate all-canon file cano.txt is excluded."
    )

manifest=[]
for p in txt_files:
    b=p.stat().st_size
    manifest.append({"relative_path": str(p.relative_to(INPUT_ROOT)), "file_name": p.name, "bytes": b})
file_manifest_df=pd.DataFrame(manifest)
write_csv(file_manifest_df, DIRS["results"] / "dataset_file_manifest.csv")

metadata_files=sorted(INPUT_ROOT.rglob("metadata.json"))
metadata_preview=None
if metadata_files:
    try:
        metadata_preview=json.loads(metadata_files[0].read_text(encoding="utf-8"))
        print("Found metadata file:", metadata_files[0].relative_to(INPUT_ROOT))
    except Exception as exc:
        print("Metadata file could not be parsed; continuing using text files only:", repr(exc))

print("Text files selected:", len(txt_files))
print("Total raw bytes:", int(file_manifest_df["bytes"].sum()))

Discovered 60 .txt files under /kaggle/input; excluded duplicate-container candidates: 0
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/A Study In Scarlet.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/The Hound of the Baskervilles.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/The Sign of the Four.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/The Valley of Fear.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/short_stories/A Case of Identity.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/short_stories/A Scandal in Bohemia.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/short_stories/Black Peter.txt
 - datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/short_stories/Charles Augustus Milverton

## Section 2 — Load, normalize and deduplicate works

In [3]:
GUTENBERG_START_RE = re.compile(r"^\s*\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG", re.I | re.M)
GUTENBERG_END_RE = re.compile(r"^\s*\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG", re.I | re.M)
CHAR_TRANSLATION = str.maketrans({
    "‘": "'", "’": "'", "‛": "'", "´": "'", "`": "'",
    "“": '"', "”": '"', "„": '"', "‟": '"',
    "—": "-", "–": "-", "−": "-", "æ": "ae", "Æ": "AE", "œ": "oe", "Œ": "OE",
    "…": "...",
    # NFKD does not decompose the German sharp-s; transliterate it explicitly
    # so the downstream ASCII word tokenizer and normalization assertion agree.
    "ß": "ss", "ẞ": "SS",
})

def strip_gutenberg(text: str) -> str:
    start = GUTENBERG_START_RE.search(text)
    if start:
        # Drop the entire start-marker line, including its title/version tail.
        line_end=text.find("\n",start.end())
        text=text[line_end+1:] if line_end>=0 else ""
    end = GUTENBERG_END_RE.search(text)
    if end:
        # Drop the end-marker line and all boilerplate following it.
        line_start=text.rfind("\n",0,end.start())
        text=text[:line_start if line_start>=0 else end.start()]
    return text

def normalise_chars(text: str) -> str:
    text = text.translate(CHAR_TRANSLATION)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"-{2,}", "-", text)
    return text

def to_paragraphs(text: str) -> list[str]:
    # Blank line ends a paragraph; hard-wrapped source lines inside paragraphs are rejoined.
    chunks = re.split(r"\n\s*\n+", text.replace("\r\n", "\n").replace("\r", "\n"))
    paragraphs=[]
    for chunk in chunks:
        para=" ".join(line.strip() for line in chunk.split("\n") if line.strip())
        para=re.sub(r"\s+", " ", para).strip()
        if para:
            paragraphs.append(para)
    return paragraphs

def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(normalise_chars(text))

def _clean_one(raw: str) -> tuple[list[str], str]:
    text = normalise_chars(strip_gutenberg(raw))
    paragraphs = to_paragraphs(text)
    clean_text = "\n\n".join(paragraphs)
    return paragraphs, clean_text

works = {}
seen_hashes = {}
empty_files=[]
duplicate_files=[]
read_errors=[]
for p in txt_files:
    try:
        raw = p.read_text(encoding="utf-8-sig")
    except UnicodeDecodeError as exc:
        read_errors.append({"path": str(p.relative_to(INPUT_ROOT)), "error": str(exc)})
        # Do not silently use mojibake. The explicit fallback is only to make the issue visible in diagnostics.
        raw = p.read_text(encoding="utf-8", errors="replace")
    paras, clean_text = _clean_one(raw)
    if not clean_text.strip():
        empty_files.append(str(p.relative_to(INPUT_ROOT)))
        continue
    digest=hashlib.sha256(normalise_chars(clean_text).encode("utf-8")).hexdigest()
    if digest in seen_hashes:
        duplicate_files.append({"duplicate": str(p.relative_to(INPUT_ROOT)), "kept": seen_hashes[digest]})
        continue
    name=str(p.relative_to(INPUT_ROOT)).replace("\\", "/")
    seen_hashes[digest]=name
    para_tokens=[tokenize(para) for para in paras]
    para_tokens=[x for x in para_tokens if x]
    if not para_tokens:
        empty_files.append(str(p.relative_to(INPUT_ROOT)))
        continue
    works[name] = {
        "path": str(p), "relative_path": str(p.relative_to(INPUT_ROOT)),
        "paragraphs": paras, "text": clean_text, "paragraph_tokens": para_tokens,
    }

if read_errors:
    print("WARNING: UTF-8 decoding needed replacement characters in", len(read_errors), "file(s). Inspect results/encoding_warnings.json.")
    write_json(read_errors, DIRS["results"] / "encoding_warnings.json")
if empty_files:
    print("Empty files dropped:", empty_files)
if duplicate_files:
    print("Exact normalized duplicate files dropped:", len(duplicate_files))
    write_json(duplicate_files, DIRS["results"] / "duplicate_files.json")
assert len(works) >= CFG.data.min_works, f"Only {len(works)} non-empty unique works remain. Re-check the attached dataset."

# Token streams include an EOS boundary between paragraphs for split-size / overlap checks.
tok_by_work = {}
work_token_counts = {}
for name, item in works.items():
    flattened=[]
    for i, para_tokens in enumerate(item["paragraph_tokens"]):
        flattened.extend(para_tokens)
        flattened.append("<eos>")
    tok_by_work[name]=flattened
    work_token_counts[name]=len(flattened)

# Gate 0 diagnostics: text shape, UTF-8 readability, markers and sample excerpts.
print(f"Loaded {len(works)} unique non-empty works; removed {len(duplicate_files)} exact duplicates.")
for name in list(sorted(works))[:2]:
    print("\nSample:", name, "\n", works[name]["text"][:300].replace("\n", " "))
assert all("*** START OF" not in w["text"].upper() for w in works.values()), "Gutenberg start boilerplate remains in a cleaned work."
assert all("*** END OF" not in w["text"].upper() for w in works.values()), "Gutenberg end boilerplate remains in a cleaned work."
assert all(sum(len(x) for x in w["paragraph_tokens"]) > 0 for w in works.values())
non_ascii_letters=[(name,ch) for name,w in works.items() for ch in w["text"] if ch.isalpha() and ord(ch)>127]
assert not non_ascii_letters, f"Non-ASCII letters remain after normalization; inspect examples: {non_ascii_letters[:10]}"

Loaded 60 unique non-empty works; removed 0 exact duplicates.

Sample: datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/A Study In Scarlet.txt 
 a study in scarlet  arthur conan doyle  table of contents  part i mr. sherlock holmes the science of deduction the lauriston garden mystery what john rance had to tell our advertisement brings a visitor tobias gregson shows what he can do light in the darkness  part ii on the great alkali plain the 

Sample: datasets/bengiefigueroa/sherlock-holmes-complete-collection/sherlock_holmes/novels/The Hound of the Baskervilles.txt 
 the hound of the baskervilles  arthur conan doyle  table of contents mr. sherlock holmes the curse of the baskervilles the problem sir henry baskerville three broken threads baskerville hall the stapletons of merripit house first report of dr. watson second report of dr. watson extract from the diar


## Section 3 — Corpus description and EDA

In [4]:
all_tokens=[]
sentence_lengths=[]
paragraph_lengths=[]
work_rows=[]
for name, item in works.items():
    text=item["text"]
    toks=[t for para in item["paragraph_tokens"] for t in para]
    all_tokens.extend(toks)
    paragraph_lengths.extend(len(pt) for pt in item["paragraph_tokens"])
    rough_sentences=[s.strip() for s in re.split(r"(?<=[.!?])\s+", normalise_chars(text)) if s.strip()]
    sentence_lengths.extend(len(TOKEN_RE.findall(s)) for s in rough_sentences if TOKEN_RE.findall(s))
    work_rows.append({"work": name, "characters": len(text), "paragraphs": len(item["paragraphs"]),
                      "lexical_tokens": len(toks), "tokens_with_eos": work_token_counts[name]})

corpus_counter=Counter(all_tokens)
total_lexical_tokens=len(all_tokens)
punct_set={".",",","!","?",";",":",'"',"-"}
stats={
    "dataset_name": "Sherlock Holmes Complete Collection",
    "dataset_source_url": "https://www.kaggle.com/datasets/bengiefigueroa/sherlock-holmes-complete-collection",
    "n_works": len(works), "total_bytes_selected": int(file_manifest_df["bytes"].sum()),
    "total_characters": int(sum(len(w["text"]) for w in works.values())),
    "total_paragraphs": int(sum(len(w["paragraphs"]) for w in works.values())),
    "total_lexical_tokens": int(total_lexical_tokens),
    "total_tokens_including_paragraph_eos": int(sum(work_token_counts.values())),
    "unique_tokens_casefolded": int(len(corpus_counter)),
    "mean_sentence_length_words_approx": float(np.mean(sentence_lengths)) if sentence_lengths else 0.0,
    "median_sentence_length_words_approx": float(np.median(sentence_lengths)) if sentence_lengths else 0.0,
    "mean_paragraph_length_words": float(np.mean(paragraph_lengths)) if paragraph_lengths else 0.0,
    "median_paragraph_length_words": float(np.median(paragraph_lengths)) if paragraph_lengths else 0.0,
    "punctuation_token_share": float(sum(corpus_counter[t] for t in punct_set)/max(1,total_lexical_tokens)),
    "top_20_tokens": [{"token": k, "count": int(v)} for k,v in corpus_counter.most_common(20)],
    "license_note": "The selected Kaggle listing is described in PLAN.md as CC0/Public Domain; re-check the live listing and license before submission or redistribution.",
}
write_csv(pd.DataFrame(work_rows), DIRS["results"] / "work_statistics.csv")
write_json(stats, DIRS["results"] / "corpus_stats.json")
print(pd.DataFrame([
    ("Works", stats["n_works"]), ("Characters", stats["total_characters"]),
    ("Paragraphs", stats["total_paragraphs"]), ("Lexical tokens", stats["total_lexical_tokens"]),
    ("Unique tokens", stats["unique_tokens_casefolded"]),
    ("Mean sentence length (approx.)", round(stats["mean_sentence_length_words_approx"],2)),
    ("Mean paragraph length", round(stats["mean_paragraph_length_words"],2)),
], columns=["Statistic","Value"]).to_string(index=False))

fig, axes=plt.subplots(1,2,figsize=(13,4.5))
frequencies=np.array(sorted(corpus_counter.values(), reverse=True), dtype=np.float64)
if len(frequencies):
    axes[0].plot(np.arange(1,len(frequencies)+1), frequencies, linewidth=1)
    axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_title("Corpus token frequency (Zipf-like plot)"); axes[0].set_xlabel("Token rank (log scale)"); axes[0].set_ylabel("Frequency (log scale)")
if sentence_lengths:
    axes[1].hist(sentence_lengths, bins=40)
axes[1].set_title("Approximate sentence length distribution"); axes[1].set_xlabel("Tokens per sentence"); axes[1].set_ylabel("Count")
fig.tight_layout(); fig.savefig(DIRS["plots"] / "corpus_eda.png", dpi=150); plt.close(fig)

                     Statistic      Value
                         Works      60.00
                    Characters 3551474.00
                    Paragraphs   15622.00
                Lexical tokens  792873.00
                 Unique tokens   19641.00
Mean sentence length (approx.)      22.07
         Mean paragraph length      50.75


## Section 4 — Work-level splits and leakage checks

In [5]:
def allocate_splits(work_tokens: dict[str,int], seed: int, targets=(0.8,0.1,0.1), bands=((.78,.82),(.08,.12),(.08,.12))):
    names=sorted(work_tokens)
    total=float(sum(work_tokens.values()))
    if total <= 0: raise ValueError("Corpus has no tokens.")
    best=None
    for split_seed in range(seed, seed + CFG.data.split_max_tries):
        order=names.copy(); random.Random(split_seed).shuffle(order)
        totals=[0,0,0]; mapping={}; current=0
        for name in order:
            n=int(work_tokens[name])
            while current < 2 and (totals[current] + n) > (targets[current]*total + n/2):
                current += 1
            mapping[name]=("train","val","test")[current]
            totals[current]+=n
        shares=[x/total for x in totals]
        deviation=max(abs(shares[i]-targets[i]) for i in range(3))
        record={"split_seed":split_seed,"shares":shares,"totals":totals,"max_deviation":deviation,
                "within_bands":all(bands[i][0] <= shares[i] <= bands[i][1] for i in range(3))}
        if best is None or deviation < best[0]: best=(deviation,mapping,record)
        if record["within_bands"]:
            return mapping, record
    # As the plan specifies, keep the lowest-deviation allocation if no seed satisfies every band.
    return best[1], best[2]

def ngram_hashes(tokens: Sequence[str], n: int) -> set[int]:
    if len(tokens) < n: return set()
    return {hash(tuple(tokens[i:i+n])) for i in range(len(tokens)-n+1)}

def overlap_by_work(names_by_split, n=12):
    train_names=names_by_split["train"]
    train_set=set()
    for name in train_names: train_set |= ngram_hashes(tok_by_work[name], n)
    details=[]
    overlap_total=0; n_total=0
    for split in ("val","test"):
        for name in names_by_split[split]:
            tokens=tok_by_work[name]
            grams=[hash(tuple(tokens[i:i+n])) for i in range(max(0,len(tokens)-n+1))]
            match=sum(g in train_set for g in grams)
            denom=len(grams)
            share=match/max(1,denom)
            details.append({"work":name,"split":split,"ngrams":denom,"overlap_ngrams":match,"overlap_share":share})
            overlap_total+=match; n_total+=denom
    split_share=overlap_total/max(1,n_total)
    max_work=max((r["overlap_share"] for r in details), default=0.0)
    return details, split_share, max_work

# Remove a held-out work only when its 12-gram overlap exceeds 30%, which the plan treats as a near duplicate.
removed_near=[]
for attempt in range(5):
    mapping, split_record=allocate_splits(work_token_counts, CFG.data.split_seed_start,
                                          CFG.data.split_targets, CFG.data.split_bands)
    names_by_split={s:sorted([w for w,sp in mapping.items() if sp==s]) for s in ("train","val","test")}
    details, split_overlap, max_work_overlap=overlap_by_work(names_by_split, CFG.data.overlap_n)
    bad=[d for d in details if d["overlap_share"] > CFG.data.overlap_work_max]
    if not bad: break
    if attempt == 4:
        raise RuntimeError("Near-duplicate overlap remains after four removal/re-split attempts; inspect leakage_check.json.")
    # Remove held-out near duplicates and rebuild the split, as specified in PLAN.md.
    for d in bad:
        nm=d["work"]
        if nm in works:
            removed_near.append({**d,"action":"removed and split rebuilt"})
            works.pop(nm); tok_by_work.pop(nm); work_token_counts.pop(nm)
    if len(works) < CFG.data.min_works:
        raise RuntimeError("Near-duplicate removal left too few works; inspect leakage_check.json and dataset inputs.")

names_by_split={s:sorted([w for w,sp in mapping.items() if sp==s]) for s in ("train","val","test")}
assert all(names_by_split[s] for s in names_by_split), "At least one split is empty."
assert len(set(names_by_split["train"]) | set(names_by_split["val"]) | set(names_by_split["test"])) == len(works)
assert not (set(names_by_split["train"]) & set(names_by_split["val"]))
assert not (set(names_by_split["train"]) & set(names_by_split["test"]))
assert not (set(names_by_split["val"]) & set(names_by_split["test"]))
details, split_overlap, max_work_overlap=overlap_by_work(names_by_split, CFG.data.overlap_n)
actual_shares={s:sum(work_token_counts[w] for w in names_by_split[s])/sum(work_token_counts.values()) for s in names_by_split}
within_bands=all(CFG.data.split_bands[i][0] <= actual_shares[s] <= CFG.data.split_bands[i][1] for i,s in enumerate(("train","val","test")))
if not within_bands:
    msg=f"Work-level split is outside target bands (train/val/test={actual_shares}); using the lowest-deviation split as specified by PLAN.md."
    print("WARNING:",msg); RUN["warnings"].append(msg)
if split_overlap >= CFG.data.overlap_split_max:
    msg=f"Split-level held-out 12-gram overlap is {split_overlap:.3%}, at or above the plan's <1% target; inspect common repeated passages before proceeding."
    RUN["warnings"].append(msg)
    raise RuntimeError(msg)
if max_work_overlap > CFG.data.overlap_work_max:
    raise RuntimeError(f"A held-out work still exceeds 30% train n-gram overlap: {max_work_overlap:.3%}. Inspect duplicate inputs.")

split_manifest=[]
for split,names in names_by_split.items():
    for name in names:
        split_manifest.append({"work":name,"split":split,"tokens":work_token_counts[name]})
write_csv(pd.DataFrame(split_manifest), DIRS["results"] / "split_manifest.csv")
# Refresh corpus-level totals if any held-out near-duplicate work was removed during leakage checks.
if removed_near:
    final_tokens=[t for item in works.values() for para in item["paragraph_tokens"] for t in para]
    final_counter=Counter(final_tokens)
    stats.update({"n_works":len(works),"total_characters":int(sum(len(w["text"]) for w in works.values())),
                  "total_paragraphs":int(sum(len(w["paragraphs"]) for w in works.values())),
                  "total_lexical_tokens":len(final_tokens),"total_tokens_including_paragraph_eos":int(sum(work_token_counts.values())),
                  "unique_tokens_casefolded":len(final_counter),"punctuation_token_share":float(sum(final_counter[t] for t in punct_set)/max(1,len(final_tokens))),
                  "top_20_tokens":[{"token":k,"count":int(v)} for k,v in final_counter.most_common(20)]})
    write_json(stats,DIRS["results"] / "corpus_stats.json")
leakage_report={
    "split_seed":split_record["split_seed"], "split_shares":actual_shares,
    "within_target_bands":within_bands, "overlap_n":CFG.data.overlap_n,
    "heldout_ngram_overlap_share":split_overlap, "max_single_heldout_work_overlap":max_work_overlap,
    "per_work":details, "near_duplicate_removals":removed_near,
    "notes":"Hashes use Python tuple hash and are compared only within this process; no hash values are saved.",
}
write_json(leakage_report, DIRS["results"] / "leakage_check.json")
RUN["split_seed"]=split_record["split_seed"]
RUN["split_shares"]=actual_shares
RUN["split_works"]={s:len(v) for s,v in names_by_split.items()}
RUN["removed_near_duplicates"]=removed_near
RUN["heldout_ngram_overlap_share"]=split_overlap
print("Split manifest:", {s:len(v) for s,v in names_by_split.items()})
print("Token shares:", {k:round(v,4) for k,v in actual_shares.items()}, "split overlap:", round(split_overlap,4))

Split manifest: {'train': 49, 'val': 8, 'test': 3}
Token shares: {'train': 0.8, 'val': 0.1055, 'test': 0.0945} split overlap: 0.0004


## Section 5 — Train-only vocabulary and encoding

In [6]:
@dataclass(frozen=True)
class Vocab:
    itos: tuple[str, ...]
    stoi: dict[str, int]
    sha256: str
    UNK: int = 0
    BOS: int = 1
    EOS: int = 2
    @property
    def V(self): return len(self.itos)

def build_vocab(train_token_lists: Iterable[Sequence[str]], max_vocab: int, min_freq: int) -> Vocab:
    counts=Counter(tok for seq in train_token_lists for tok in seq if tok not in {"<eos>","<bos>","<unk>"})
    kept=[(tok,count) for tok,count in counts.items() if count >= min_freq]
    kept.sort(key=lambda z:(-z[1],z[0]))
    kept=kept[:max_vocab-3]
    itos=("<unk>","<bos>","<eos>")+tuple(tok for tok,_ in kept)
    vocab_hash=hashlib.sha256("\n".join(itos).encode("utf-8")).hexdigest()
    return Vocab(itos=itos, stoi={tok:i for i,tok in enumerate(itos)}, sha256=vocab_hash)

def encode(text: str, vocab: Vocab, add_eos: bool=False) -> list[int]:
    ids=[vocab.stoi.get(tok,vocab.UNK) for tok in tokenize(text)]
    if add_eos: ids.append(vocab.EOS)
    return ids

def encode_work(paragraphs: list[str], vocab: Vocab) -> list[int]:
    ids=[]
    for para in paragraphs:
        ids.extend(encode(para, vocab, add_eos=True))
    return ids

def detokenize(ids: Sequence[int], vocab: Vocab) -> str:
    toks=[]
    for idx in ids:
        idx=int(idx)
        if idx == vocab.BOS: continue
        if idx == vocab.UNK: tok="<unk>"
        else: tok=vocab.itos[idx]
        if tok == "<eos>":
            if toks and toks[-1] != "\n": toks.append("\n")
        else: toks.append(tok)
    text=" ".join(t for t in toks if t != "\n")
    # Restore normal English spacing around punctuation and quotation marks.
    text=re.sub(r"\s+([.,!?;:])", r"\1", text)
    text=re.sub(r"\s+\-\s+", " - ", text)
    text=text.replace('" ', '"').replace(' "','"')
    return text.strip()

train_token_lists=[]
for w in names_by_split["train"]:
    train_token_lists.extend(works[w]["paragraph_tokens"])
vocab=build_vocab(train_token_lists, CFG.data.max_vocab, CFG.data.min_freq)
assert vocab.itos[:3] == ("<unk>","<bos>","<eos>")
assert all(vocab.stoi[tok] == i for i,tok in enumerate(vocab.itos))
assert vocab.V <= CFG.data.max_vocab
expected_v = min(CFG.data.max_vocab, 3 + sum(1 for tok,c in Counter(t for seq in train_token_lists for t in seq).items() if c >= CFG.data.min_freq))
assert vocab.V == expected_v, f"Vocabulary size mismatch: got {vocab.V}, expected {expected_v}"

# ids_by_work are encoded only after the train-only vocabulary has been fixed.
ids_by_work={name:encode_work(item["paragraphs"],vocab) for name,item in works.items()}
assert all(vocab.BOS not in ids for ids in ids_by_work.values())
assert all(all(0 <= idx < vocab.V for idx in ids) for ids in ids_by_work.values())
# Cross-check that the shared encoder and paragraph encoding agree, including EOS.
for name in list(sorted(works))[:min(20,len(works))]:
    direct=[]
    for para in works[name]["paragraphs"]: direct.extend(encode(para,vocab,add_eos=True))
    assert direct == ids_by_work[name], f"Encoding path mismatch for {name}"

is_word_np=np.array([bool(WORD_RE.fullmatch(tok)) if i >= 3 else False for i,tok in enumerate(vocab.itos)],dtype=bool)
is_word=torch.tensor(is_word_np,dtype=torch.bool,device=device)

# Save a canonical payload. sha256 identifies the ordered vocabulary (newline-delimited tokens);
# it is not a self-referential hash of a JSON file that contains its own hash.
train_frequency=Counter(tok for seq in train_token_lists for tok in seq)
vocab_payload={"itos":list(vocab.itos),"V":vocab.V,"sha256":vocab.sha256,
               "min_freq":CFG.data.min_freq,"max_vocab":CFG.data.max_vocab,
               "train_counts_for_kept_tokens":{tok:int(train_frequency.get(tok,0)) for tok in vocab.itos[3:]},
               "special_tokens":{"<unk>":0,"<bos>":1,"<eos>":2}}
write_json(vocab_payload, DIRS["results"] / "vocab.json")

unk_rates={}
share_kept={}
for split,names in names_by_split.items():
    raw_count=sum(work_token_counts[w] for w in names)
    unk_count=sum(ids_by_work[w].count(vocab.UNK) for w in names)
    unk_rates[split]=unk_count/max(1,raw_count)
    if split == "train":
        n_lex=sum(sum(len(p) for p in works[w]["paragraph_tokens"]) for w in names)
        n_kept=sum(sum(idx != vocab.UNK for idx in ids_by_work[w]) for w in names)
        share_kept[split]=n_kept/max(1,raw_count)
    else:
        share_kept[split]=None
stats.update({"vocab_size":vocab.V,"vocab_sha256":vocab.sha256,"train_vocab_token_share_kept_including_eos":share_kept["train"],
              "unk_rate_per_split_including_eos":unk_rates,"vocab_min_freq":CFG.data.min_freq,"vocab_max_size":CFG.data.max_vocab})
write_json(stats, DIRS["results"] / "corpus_stats.json")
assert unk_rates["train"] < 0.2, f"Training <unk> rate {unk_rates['train']:.3f} exceeds the plan's 0.20 sanity ceiling. Inspect vocabulary/data."
print("Vocab size:",vocab.V,"| hash:",vocab.sha256)
print("<unk> rate per split:",{k:round(v,4) for k,v in unk_rates.items()})

Vocab size: 8600 | hash: 509b3e76a46b5676cd882b5afa6f4e5f80cecaf2d418c9e0d8a1f015397bab99
<unk> rate per split: {'train': 0.0183, 'val': 0.0348, 'test': 0.0415}


## Section 6 — Build streams, target positions, window loaders

In [7]:
def build_stream(ids_per_work: list[list[int]], L: int, bos: int):
    stream=[]; positions=[]; work_index=[]; starts=[]
    cursor=0
    for wi, ids in enumerate(ids_per_work):
        if not ids: continue
        stream.extend([bos]*L); work_index.extend([wi]*L)
        starts.append(cursor+L)
        positions.extend(range(cursor+L,cursor+L+len(ids)))
        stream.extend(ids); work_index.extend([wi]*len(ids))
        cursor += L+len(ids)
    return (torch.tensor(stream,dtype=torch.long), torch.tensor(positions,dtype=torch.long),
            torch.tensor(work_index,dtype=torch.long), torch.tensor(starts,dtype=torch.long))

def gather_windows(stream: torch.Tensor, pos: torch.Tensor, L: int) -> torch.Tensor:
    offsets=torch.arange(-L,0,device=pos.device,dtype=torch.long)
    return stream[pos[:,None]+offsets[None,:]]

def leaky_gather(stream: torch.Tensor, pos: torch.Tensor, L: int) -> torch.Tensor:
    # Deliberately incorrect positive control: includes the target at the right edge.
    offsets=torch.arange(-L+1,1,device=pos.device,dtype=torch.long)
    return stream[pos[:,None]+offsets[None,:]]

class WindowLoader:
    def __init__(self, stream, positions, L, batch_size, shuffle, seed=None, device=None):
        self.stream=stream; self.positions=positions; self.L=int(L); self.batch_size=int(batch_size)
        self.shuffle=bool(shuffle); self.seed=int(seed or 0); self.device=device or stream.device
        self.order=None; self._epoch=0; self.set_epoch(0)
    def set_epoch(self, epoch_index: int) -> None:
        self._epoch=int(epoch_index)
        if self.shuffle:
            g=torch.Generator(device="cpu").manual_seed(self.seed+self._epoch)
            order=torch.randperm(len(self.positions),generator=g,dtype=torch.long)
            self.order=order.to(self.device)
        else:
            self.order=torch.arange(len(self.positions),device=self.device,dtype=torch.long)
    def perm_digest(self) -> str:
        arr=self.order[:min(1024,len(self.order))].detach().cpu().numpy().astype(np.int64,copy=False)
        return hashlib.sha1(arr.tobytes()).hexdigest()[:16]
    def __len__(self): return math.ceil(len(self.positions)/self.batch_size)
    def __iter__(self):
        for start in range(0,len(self.order),self.batch_size):
            ix=self.order[start:start+self.batch_size]
            pos=self.positions[ix]
            x=gather_windows(self.stream,pos,self.L)
            y=self.stream[pos]
            yield x,y

@dataclass
class DataBundle:
    vocab: Vocab
    is_word: torch.Tensor
    streams: dict
    positions: dict
    positions_full: dict
    work_index: dict
    starts: dict
    train_counts: torch.Tensor
    loaders: dict
    ids_by_work_split: dict

streams_cpu={}; positions_cpu={}; work_index_cpu={}; starts_cpu={}
for split in ("train","val","test"):
    ordered=sorted(names_by_split[split])
    stream, positions, wi, starts=build_stream([ids_by_work[w] for w in ordered],CFG.data.L,vocab.BOS)
    streams_cpu[split]=stream; positions_cpu[split]=positions; work_index_cpu[split]=wi; starts_cpu[split]=starts

N_train_full=len(positions_cpu["train"])
if N_train_full > 1_500_000 and CFG.data.train_stride == 1:
    CFG=replace(CFG,data=replace(CFG.data,train_stride=2))
    RUN["cfg_changes"].append({"change":"train_stride=2","reason":"train token count exceeded 1.5 million size guard"})
if N_train_full < 300_000:
    RUN["small_corpus"]=True

streams={s:streams_cpu[s].to(device) for s in streams_cpu}
positions_full={s:positions_cpu[s].to(device) for s in positions_cpu}
positions={s:(positions_full[s][::CFG.data.train_stride] if s=="train" else positions_full[s]) for s in positions_full}
work_index={s:work_index_cpu[s] for s in work_index_cpu}  # CPU to facilitate structural checks
starts={s:starts_cpu[s].to(device) for s in starts_cpu}

train_counts=torch.bincount(streams["train"][positions_full["train"]],minlength=vocab.V)
assert train_counts.shape == (vocab.V,)
assert int(train_counts[vocab.BOS].item()) == 0
assert int(train_counts[vocab.EOS].item()) > 0
assert int(train_counts[vocab.UNK].item()) > 0, "The plan expects a nonzero training <unk> count for the unsmoothed unigram baseline. Inspect the corpus/vocabulary."

# Fixed train evaluation subsample from stride-adjusted target positions.
g_sub=torch.Generator(device="cpu").manual_seed(CFG.train.subsample_seed)
sub_n=min(CFG.train.subsample_n,len(positions["train"]))
sub_idx=torch.randperm(len(positions["train"]),generator=g_sub)[:sub_n].sort().values.to(device)
sub_positions=positions["train"][sub_idx]

def make_loaders(cfg: Config, stream_dict, pos_dict, train_sub_positions, seed: int):
    return {
        "train": WindowLoader(stream_dict["train"],pos_dict["train"],cfg.data.L,cfg.train.batch_size,True,seed,device),
        "val": WindowLoader(stream_dict["val"],pos_dict["val"],cfg.data.L,cfg.train.eval_batch_size,False,seed,device),
        "test": WindowLoader(stream_dict["test"],pos_dict["test"],cfg.data.L,cfg.train.eval_batch_size,False,seed,device),
        "train_sub": WindowLoader(stream_dict["train"],train_sub_positions,cfg.data.L,cfg.train.eval_batch_size,False,seed,device),
    }

loaders=make_loaders(CFG,streams,positions,sub_positions,CFG.run.model_seed)
data=DataBundle(vocab=vocab,is_word=is_word,streams=streams,positions=positions,positions_full=positions_full,
                work_index=work_index,starts=starts,train_counts=train_counts,loaders=loaders,
                ids_by_work_split={s:{w:ids_by_work[w] for w in names_by_split[s]} for s in names_by_split})

# Structural/index checks. Validation/test remain stride 1 and evaluate every real target once.
L=CFG.data.L
offs=torch.arange(-L,0,device=device)
assert len(offs)==L and int(offs[0])==-L and int(offs[-1])==-1
for split in ("train","val","test"):
    assert positions_full[split].numel() == sum(len(ids_by_work[w]) for w in names_by_split[split])
    for first_target in starts[split]:
        assert torch.all(streams[split][first_target-L:first_target] == vocab.BOS), f"{split}: work does not begin with an all-BOS context."
    assert int(positions_full[split].min()) >= L
    assert int(positions_full[split].max()) < streams[split].numel()
    if split in ("val","test"):
        assert positions[split].numel() == positions_full[split].numel()
    if len(positions[split]):
        all_wi=work_index[split].numpy()
        all_pos=positions[split]
        for start_i in range(0,len(all_pos),4096):
            check_ix=all_pos[start_i:start_i+4096]
            assert torch.all(check_ix[:,None] + offs[None,:] < check_ix[:,None])
            assert torch.all(streams[split][check_ix] != vocab.BOS)
            check_cpu=check_ix.detach().cpu().numpy()
            window_ix=check_cpu[:,None] + np.arange(-L,0,dtype=np.int64)[None,:]
            assert np.all(all_wi[window_ix] == all_wi[check_cpu,None]), f"{split}: a context window crosses a work boundary."

# Shift identity on adjacent validation targets from the same work.
posv=positions["val"]
if len(posv)>1:
    prev=posv[:-1][posv[1:] == posv[:-1]+1]
    if len(prev):
        prev=prev[:min(2048,len(prev))]
        x0=gather_windows(streams["val"],prev,L)
        x1=gather_windows(streams["val"],prev+1,L)
        y0=streams["val"][prev]
        assert torch.equal(x1,torch.cat([x0[:,1:],y0[:,None]],dim=1)), "Window shift identity failed."

assert len(loaders["train"]) == math.ceil(len(positions["train"])/CFG.train.batch_size)
assert len(loaders["val"]) == math.ceil(len(positions["val"])/CFG.train.eval_batch_size)
xa,ya=next(iter(loaders["val"]))
assert xa.shape[1:] == (L,) and ya.ndim==1 and xa.dtype==torch.long and ya.dtype==torch.long
assert xa.device == device and ya.device == device

RUN["N_by_split"]={s:int(len(positions_full[s])) for s in positions_full}
RUN["stream_lengths"]={s:int(streams[s].numel()) for s in streams}
RUN["train_stride"]=CFG.data.train_stride
RUN["window_counts_after_stride"]={s:int(len(positions[s])) for s in positions}
RUN["vocab_size"]=vocab.V
RUN["parameter_counts"]={}
print(pd.DataFrame([{"split":s,"works":len(names_by_split[s]),"target_tokens":len(positions_full[s]),
                     "windows_used":len(positions[s]),"stream_length":streams[s].numel()} for s in ("train","val","test")]).to_string(index=False))
print("Train stride:",CFG.data.train_stride,"| Example batch:",tuple(xa.shape),tuple(ya.shape))

split  works  target_tokens  windows_used  stream_length
train     49         646757        646757         648227
  val      8          85323         85323          85563
 test      3          76415         76415          76505
Train stride: 1 | Example batch: (1024, 30) (1024,)


## Section 7 — Four model architectures and leakage smoke tests

In [8]:
class RecurrentLM(nn.Module):
    cell="rnn"; bidirectional=False
    def __init__(self,vocab_size:int,embed_dim:int,hidden_size:int,num_layers:int,dropout:float):
        super().__init__()
        self.vocab_size=int(vocab_size); self.embed_dim=int(embed_dim); self.hidden_size=int(hidden_size)
        self.num_layers=int(num_layers); self.dropout=float(dropout)
        self.directions=2 if self.bidirectional else 1
        self.embedding=nn.Embedding(self.vocab_size,self.embed_dim)
        self.emb_drop=nn.Dropout(self.dropout)
        self.rnn=self._make_rnn()
        self.head_drop=nn.Dropout(self.dropout)
        self.head=nn.Linear(self.directions*self.hidden_size,self.vocab_size)
    def _make_rnn(self): raise NotImplementedError
    def _run(self,e:torch.Tensor)->torch.Tensor: raise NotImplementedError
    def _context(self,h_n:torch.Tensor)->torch.Tensor: raise NotImplementedError
    def forward(self,x:torch.Tensor)->torch.Tensor:
        if x.ndim != 2: raise ValueError(f"Expected x shape (B,L), got {tuple(x.shape)}")
        if x.dtype != torch.long: raise TypeError(f"Expected int64 token ids, got {x.dtype}")
        e=self.emb_drop(self.embedding(x))
        h_n=self._run(e)
        c=self._context(h_n)
        return self.head(self.head_drop(c))

class SimpleRNNLM(RecurrentLM):
    cell="rnn"; bidirectional=False
    def _make_rnn(self):
        return nn.RNN(self.embed_dim,self.hidden_size,self.num_layers,nonlinearity="tanh",batch_first=True,dropout=self.dropout)
    def _run(self,e):
        _,h_n=self.rnn(e); return h_n
    def _context(self,h_n): return h_n[-1]

class BiRNNLM(RecurrentLM):
    cell="rnn"; bidirectional=True
    def _make_rnn(self):
        return nn.RNN(self.embed_dim,self.hidden_size,self.num_layers,nonlinearity="tanh",batch_first=True,dropout=self.dropout,bidirectional=True)
    def _run(self,e):
        _,h_n=self.rnn(e); return h_n
    def _context(self,h_n): return torch.cat([h_n[-2],h_n[-1]],dim=1)

class LSTMLM(RecurrentLM):
    cell="lstm"; bidirectional=False
    def _make_rnn(self):
        return nn.LSTM(self.embed_dim,self.hidden_size,self.num_layers,batch_first=True,dropout=self.dropout)
    def _run(self,e):
        _,(h_n,_)=self.rnn(e); return h_n
    def _context(self,h_n): return h_n[-1]

class GRULM(RecurrentLM):
    cell="gru"; bidirectional=False
    def _make_rnn(self):
        return nn.GRU(self.embed_dim,self.hidden_size,self.num_layers,batch_first=True,dropout=self.dropout)
    def _run(self,e):
        _,h_n=self.rnn(e); return h_n
    def _context(self,h_n): return h_n[-1]

MODEL_CLASSES={"simple_rnn":SimpleRNNLM,"birnn":BiRNNLM,"lstm":LSTMLM,"gru":GRULM,"birnn_matched":BiRNNLM}
def expected_params(g:int,D:int,V:int,E:int,H:int)->int:
    return int(V*E + D*(g*H*(E+H)+2*g*H) + D*(g*H*(D*H+H)+2*g*H) + D*H*V + V)
def count_params(model:nn.Module)->int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def matched_hidden(V:int,E:int,target_params:int)->int:
    candidates=range(256,641,8)
    return min(candidates,key=lambda h:(abs(expected_params(1,2,V,E,h)-target_params),h))
def hidden_size_for(name:str,cfg:Config,V:int)->int:
    if name=="birnn_matched":
        target=expected_params(4,1,V,cfg.model.embed_dim,cfg.model.hidden_size)
        return matched_hidden(V,cfg.model.embed_dim,target)
    return cfg.model.hidden_size
def build_model(name:str,cfg:Config,V:int)->RecurrentLM:
    if name not in MODEL_CLASSES: raise KeyError(f"Unknown model: {name}")
    h=hidden_size_for(name,cfg,V)
    return MODEL_CLASSES[name](V,cfg.model.embed_dim,h,cfg.model.num_layers,cfg.model.dropout)

# Shape / parameter smoke test using the exact four planned architectures.
set_seed(CFG.run.model_seed)
xb=torch.randint(0,vocab.V,(8,CFG.data.L),dtype=torch.long,device=device)
param_rows=[]
model_smoke={}
for name in MODEL_ORDER+("birnn_matched",):
    set_seed(CFG.run.model_seed)
    model=build_model(name,CFG,vocab.V).to(device).eval()
    with torch.no_grad():
        e=model.emb_drop(model.embedding(xb))
        out,h_n=model.rnn(e) if name!="lstm" else (lambda z:(z[0],z[1][0]))(model.rnn(e))
        c=model._context(h_n)
        logits=model(xb)
    D=2 if name in ("birnn","birnn_matched") else 1
    gates=1 if name in ("simple_rnn","birnn","birnn_matched") else (4 if name=="lstm" else 3)
    H=hidden_size_for(name,CFG,vocab.V)
    expected=expected_params(gates,D,vocab.V,CFG.model.embed_dim,H)
    actual=count_params(model)
    assert logits.shape==(8,vocab.V), f"{name}: logits shape {tuple(logits.shape)}"
    assert c.shape==(8,D*H), f"{name}: context shape {tuple(c.shape)}"
    if D == 1:
        assert torch.allclose(c,out[:,-1,:],atol=1e-6,rtol=0), f"{name}: context does not equal top-layer final output."
    else:
        expected_context=torch.cat([out[:,-1,:H],out[:,0,H:]],dim=1)
        assert torch.allclose(c,expected_context,atol=1e-6,rtol=0), f"{name}: BiRNN context directions are misaligned."
    assert actual==expected, f"{name}: parameter count {actual}, expected {expected}"
    assert list(inspect.signature(model.forward).parameters)==["x"], f"{name}: forward must accept only x"
    with torch.no_grad():
        l1=model(xb); l2=model(xb)
    assert torch.allclose(l1,l2,atol=1e-6,rtol=0), f"{name}: eval mode is not deterministic"
    param_rows.append({"name":name,"params":actual,"expected":expected,"hidden_size":H,"directions":D})
    model_smoke[name]=model
    print(name, "embedding",tuple(e.shape),"recurrent output",tuple(out.shape),"hidden",tuple(h_n.shape),"context",tuple(c.shape),"logits",tuple(logits.shape))

param_table=pd.DataFrame(param_rows)
write_csv(param_table,DIRS["results"] / "parameter_counts.csv")
print(param_table.to_string(index=False))
RUN["parameter_counts"]={r["name"]:int(r["params"]) for r in param_rows}

# Verify train/eval dropout semantics; dropout should vary in train mode and be disabled in eval mode.
for name,model in model_smoke.items():
    model.train()
    a=model(xb); b=model(xb)
    assert not torch.allclose(a,b,atol=1e-7,rtol=0), f"{name}: dropout appears inactive in train mode"
    model.eval()
    with torch.no_grad():
        a=model(xb); b=model(xb)
    assert torch.allclose(a,b,atol=1e-6,rtol=0), f"{name}: eval mode changed across identical calls"

# Positive-control leakage test. The correct gather uses positions [p-L, p), excluding target p.
# The deliberately wrong gather includes p and must change when p is perturbed.
def future_perturbation_test(model,split="val",n_positions=32):
    stream=streams[split]
    poss=positions_full[split]
    n=min(n_positions,len(poss))
    g=torch.Generator(device="cpu").manual_seed(999)
    chosen=poss[torch.randperm(len(poss),generator=g)[:n].to(device)]
    model.eval()
    for p_tensor in chosen:
        p=int(p_tensor.item())
        original=gather_windows(stream,p_tensor.view(1),CFG.data.L)
        modified=stream.clone()
        # Change the target itself and randomize every later token.
        modified[p]=(modified[p]+1) % vocab.V
        if p+1 < modified.numel():
            tail=torch.randint(0,vocab.V,(modified.numel()-p-1,),generator=g,dtype=torch.long).to(device)
            modified[p+1:]=tail
        correct=gather_windows(modified,p_tensor.view(1),CFG.data.L)
        assert torch.equal(original,correct), "Correct gather changed when target/future stream values were perturbed."
        with torch.no_grad():
            l0=model(original); l1=model(correct)
        assert torch.allclose(l0,l1,atol=1e-6,rtol=0), "Model logits changed even though correct context window was unchanged."
        wrong0=leaky_gather(stream,p_tensor.view(1),CFG.data.L)
        wrong1=leaky_gather(modified,p_tensor.view(1),CFG.data.L)
        assert not torch.equal(wrong0,wrong1), "Positive control failed: leaky gather did not include the perturbed target."
    return n

for name,model in model_smoke.items():
    n_checked=future_perturbation_test(model)
    print(f"{name}: target/future perturbation and leaky positive control passed ({n_checked} positions)")

# Initial loss check: random logits should be near ln(V), not pathologically far away.
for name,model in model_smoke.items():
    model.eval()
    with torch.no_grad():
        x0,y0=next(iter(loaders["val"]))
        init_loss=float(F.cross_entropy(model(x0[:min(64,len(x0))]),y0[:min(64,len(y0))]).item())
    if abs(init_loss-math.log(vocab.V)) > 0.5:
        msg=f"{name} initial loss {init_loss:.3f} is more than 0.5 from ln(V)={math.log(vocab.V):.3f}; inspect initialization and model output scale."
        print("WARNING:",msg); RUN["warnings"].append(msg)
    else:
        print(f"{name}: initial loss {init_loss:.3f}, ln(V)={math.log(vocab.V):.3f}")

del model_smoke, model
if torch.cuda.is_available(): torch.cuda.empty_cache()

simple_rnn embedding (8, 30, 256) recurrent output (8, 30, 512) hidden (2, 8, 512) context (8, 512) logits (8, 8600)
birnn embedding (8, 30, 256) recurrent output (8, 30, 1024) hidden (4, 8, 512) context (8, 1024) logits (8, 8600)
lstm embedding (8, 30, 256) recurrent output (8, 30, 512) hidden (2, 8, 512) context (8, 512) logits (8, 8600)
gru embedding (8, 30, 256) recurrent output (8, 30, 512) hidden (2, 8, 512) context (8, 512) logits (8, 8600)
birnn_matched embedding (8, 30, 256) recurrent output (8, 30, 784) hidden (4, 8, 392) context (8, 784) logits (8, 8600)
         name   params  expected  hidden_size  directions
   simple_rnn  7532952   7532952          512           1
        birnn 13379992  13379992          512           2
         lstm 10291608  10291608          512           1
          gru  9372056   9372056          512           1
birnn_matched 10385752  10385752          392           2
simple_rnn: target/future perturbation and leaky positive control passed (32 pos

## Section 8 — Training/evaluation harness, sanity tests and time guard

In [9]:
def batch_metric_sums(logits:torch.Tensor,y:torch.Tensor,is_word:torch.Tensor,unk_id:int,k:int=5):
    assert logits.ndim==2 and y.ndim==1 and logits.shape[0]==y.shape[0]
    assert logits.shape[1] > 0 and y.dtype==torch.long
    n=y.numel(); k=min(int(k),int(logits.shape[1]))
    nll=F.cross_entropy(logits,y,reduction="sum").double()
    top_indices=logits.topk(k,dim=1).indices
    pred=top_indices[:,0]
    word_mask=is_word[y]
    return {
        "nll_sum":nll,
        "top1_correct":(pred==y).sum().double(),
        "topk_correct":(top_indices==y[:,None]).any(dim=1).sum().double(),
        "word_correct":((pred==y)&word_mask).sum().double(),
        "word_n":word_mask.sum().double(),
        "unk_n":(y==int(unk_id)).sum().double(),
        "n":torch.tensor(float(n),device=y.device,dtype=torch.float64),
    }

def evaluate(model,loader,is_word,unk_id,k=5):
    previous_mode=model.training
    model.eval()
    sums=None
    with torch.no_grad():
        for x,y in loader:
            logits=model(x)
            batch=batch_metric_sums(logits,y,is_word,unk_id,k=k)
            if sums is None: sums={key:value.clone() for key,value in batch.items()}
            else:
                for key,value in batch.items(): sums[key]+=value
    model.train(previous_mode)
    if sums is None: raise ValueError("Cannot evaluate an empty loader.")
    n=int(sums["n"].item())
    loss=float((sums["nll_sum"]/sums["n"]).item())
    return {
        "N":n,"loss":loss,"ppl":float(math.exp(min(loss,80.0))),
        "top1":float((sums["top1_correct"]/sums["n"]).item()),
        "top5":float((sums["topk_correct"]/sums["n"]).item()),
        "top1_words_only":float((sums["word_correct"]/sums["word_n"]).item()) if sums["word_n"].item()>0 else float("nan"),
        "n_word_targets":int(sums["word_n"].item()),
        "unk_target_rate":float((sums["unk_n"]/sums["n"]).item()),
    }

def save_checkpoint(path, **payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    torch.save(payload,path)

def load_checkpoint(path,full:bool=False):
    try:
        return torch.load(path,map_location=device,weights_only=not full)
    except TypeError:  # older torch without weights_only argument
        return torch.load(path,map_location=device)

def _sync_device():
    if device.type=="cuda": torch.cuda.synchronize(device)

def _metric_toy_test():
    logits=torch.tensor([[4.,1.,0.,-1.,-2.],[0.,1.,4.,3.,2.]],device=device)
    yy=torch.tensor([0,3],dtype=torch.long,device=device)
    word_mask=torch.tensor([False,False,False,True,True],dtype=torch.bool,device=device)
    got=batch_metric_sums(logits,yy,word_mask,unk_id=0,k=5)
    assert int(got["n"].item())==2
    assert int(got["top1_correct"].item())==1
    assert int(got["topk_correct"].item())==2
    assert int(got["word_n"].item())==1 and int(got["word_correct"].item())==0
    expected_nll=F.cross_entropy(logits,yy,reduction="sum").double()
    assert torch.allclose(got["nll_sum"],expected_nll,atol=1e-10,rtol=0)
    print("Toy metric test passed: summed cross-entropy, top-1, top-5 and word-only counts.")
_metric_toy_test()

# Small-subset overfit sanity test. This is diagnostic, not one of the reported experimental runs.
def overfit_sanity(name,cfg,data,steps=300,subset_n=512,batch_size=64):
    positions_train=data.positions["train"]
    n=min(subset_n,len(positions_train))
    if n<batch_size: raise RuntimeError("Training split is too small for the planned overfit sanity check.")
    subset=positions_train[:n]
    sanity_cfg=replace(cfg,model=replace(cfg.model,dropout=0.0))
    set_seed(8800)
    model=build_model(name,sanity_cfg,data.vocab.V).to(device)
    opt=Adam(model.parameters(),lr=0.003,weight_decay=0.0)
    x_all=gather_windows(data.streams["train"],subset,cfg.data.L)
    y_all=data.streams["train"][subset]
    model.eval()
    with torch.no_grad(): initial=float(F.cross_entropy(model(x_all),y_all).item())
    model.train()
    g=torch.Generator(device="cpu").manual_seed(8800)
    order=torch.randperm(n,generator=g).to(device)
    ptr=0
    for step in range(steps):
        if ptr+batch_size>n:
            order=torch.randperm(n,generator=g).to(device); ptr=0
        ix=order[ptr:ptr+batch_size]; ptr += batch_size
        xb=x_all[ix]; yb=y_all[ix]
        opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(model(xb),yb)
        loss.backward(); clip_grad_norm_(model.parameters(),1.0); opt.step()
    model.eval()
    with torch.no_grad(): final=float(F.cross_entropy(model(x_all),y_all).item())
    del model,opt,x_all,y_all
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    threshold=0.5*math.log(data.vocab.V)
    return {"model":name,"n_examples":n,"steps":steps,"initial_loss":initial,"final_loss":final,"threshold":threshold,"passed":bool(final<=threshold)}

print("Running fixed-subset overfit sanity checks (4 architectures, 300 optimizer steps each).")
overfit_rows=[]
for name in MODEL_ORDER:
    row=overfit_sanity(name,CFG,data)
    overfit_rows.append(row)
    print(row)
write_csv(pd.DataFrame(overfit_rows),DIRS["results"] / "overfit_sanity.csv")
failed_overfit=[r for r in overfit_rows if not r["passed"]]
if failed_overfit:
    msg="Overfit sanity threshold was not met for: " + ", ".join(r["model"] for r in failed_overfit) + ". Inspect learning curves/labels before treating final results as reliable."
    RUN["warnings"].append(msg); print("WARNING:",msg)
    # Keep the notebook inspectable; this check is explicitly logged and does not fabricate a passing result.

# Benchmark each architecture: 20 warmup steps, 200 timed train steps, first 50 validation batches.
def benchmark_model(name,cfg,data,steps=200,warmup=20):
    model=build_model(name,cfg,data.vocab.V).to(device)
    opt=Adam(model.parameters(),lr=cfg.train.lr,weight_decay=cfg.train.weight_decay)
    loader=WindowLoader(data.streams["train"],data.positions["train"],cfg.data.L,cfg.train.batch_size,True,cfg.run.model_seed,device)
    loader.set_epoch(0); iterator=iter(loader)
    def next_batch():
        nonlocal iterator
        try: return next(iterator)
        except StopIteration:
            loader.set_epoch(loader._epoch+1); iterator=iter(loader); return next(iterator)
    model.train()
    for _ in range(warmup):
        x,y=next_batch(); opt.zero_grad(set_to_none=True); loss=F.cross_entropy(model(x),y); loss.backward(); clip_grad_norm_(model.parameters(),cfg.train.clip_norm); opt.step()
    _sync_device(); t0=time.perf_counter()
    for _ in range(steps):
        x,y=next_batch(); opt.zero_grad(set_to_none=True); loss=F.cross_entropy(model(x),y); loss.backward(); clip_grad_norm_(model.parameters(),cfg.train.clip_norm); opt.step()
    _sync_device(); elapsed=time.perf_counter()-t0
    model.eval(); batch_times=[]
    with torch.no_grad():
        _sync_device(); vt=time.perf_counter()
        for i,(x,y) in enumerate(data.loaders["val"]):
            if i>=50: break
            _=model(x)
        _sync_device(); val_elapsed=time.perf_counter()-vt
    n_val_batches=min(50,len(data.loaders["val"]))
    avg_val_batch=val_elapsed/max(1,n_val_batches)
    row={"model":name,"step_s":elapsed/steps,"timed_steps":steps,"warmup_steps":warmup,
         "val_batches_timed":n_val_batches,"val_batch_s":avg_val_batch,"val_50_batches_s":val_elapsed,
         "gpu_name":gpu_name}
    del model,opt,loader,iterator
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

def estimate_total_hours(bench,cfg,data):
    steps_per_epoch=math.ceil(len(data.positions["train"])/cfg.train.batch_size)
    val_batches=len(data.loaders["val"])
    sub_batches=len(data.loaders["train_sub"])
    secs=0.0
    for name in MODEL_ORDER:
        b=bench[name]
        full_val=b["val_batch_s"]*val_batches
        sub_val=b["val_batch_s"]*sub_batches
        epoch=steps_per_epoch*b["step_s"]+full_val+sub_val
        secs += cfg.train.max_epochs*epoch
    return secs/3600.0

def _rebuild_loaders_after_cfg_change():
    global sub_positions,loaders
    positions["train"]=positions_full["train"][::CFG.data.train_stride]
    g_sub=torch.Generator(device="cpu").manual_seed(CFG.train.subsample_seed)
    sub_n=min(CFG.train.subsample_n,len(positions["train"]))
    sub_idx=torch.randperm(len(positions["train"]),generator=g_sub)[:sub_n].sort().values.to(device)
    sub_positions=positions["train"][sub_idx]
    loaders=make_loaders(CFG,streams,positions,sub_positions,CFG.run.model_seed)
    data.positions=positions; data.loaders=loaders

def apply_time_guard(bench,cfg,data):
    global CFG
    log=[]
    estimate=estimate_total_hours(bench,cfg,data)
    log.append({"estimate_hours":estimate,"action":"initial estimate"})
    if estimate>cfg.run.time_budget_h and cfg.data.train_stride==1:
        CFG=replace(CFG,data=replace(CFG.data,train_stride=2))
        CFG=replace(CFG,train=replace(CFG.train,))
        RUN["cfg_changes"].append({"change":"train_stride=2","reason":"estimated full training time exceeded 6-hour budget"})
        _rebuild_loaders_after_cfg_change(); estimate=estimate_total_hours(bench,CFG,data)
        log.append({"estimate_hours":estimate,"action":"train stride set to 2 for all models"})
    if estimate>CFG.run.time_budget_h and CFG.train.max_epochs>12:
        CFG=replace(CFG,train=replace(CFG.train,max_epochs=12))
        RUN["cfg_changes"].append({"change":"max_epochs=12","reason":"estimated full training time still exceeded 6-hour budget"})
        estimate=estimate_total_hours(bench,CFG,data)
        log.append({"estimate_hours":estimate,"action":"max epochs set to 12 for all models"})
    if estimate>CFG.run.time_budget_h and CFG.model.hidden_size>384:
        CFG=replace(CFG,model=replace(CFG.model,hidden_size=384))
        RUN["cfg_changes"].append({"change":"hidden_size=384","reason":"estimated full training time still exceeded 6-hour budget"})
        estimate=estimate_total_hours(bench,CFG,data)
        log.append({"estimate_hours":estimate,"action":"hidden size set to 384 for all models"})
    if estimate>CFG.run.time_budget_h:
        msg=f"Estimated complete training is still {estimate:.2f} hours after all plan-approved time guards. Reduce the scope or use a faster Kaggle GPU before training."
        RUN["warnings"].append(msg); log.append({"estimate_hours":estimate,"action":"budget remains exceeded; stop before expensive training"})
        raise RuntimeError(msg)
    RUN["guard_log"]=log; RUN["estimated_worst_case_hours"]=estimate
    RUN["cfg_changes"]=RUN["cfg_changes"]
    print("Time-guard log:")
    for item in log: print(" -",item)
    return CFG,log

print("Benchmarking 4 model types; 20 warmup + 200 timed updates per architecture.")
benchmark_rows=[]
for name in MODEL_ORDER:
    row=benchmark_model(name,CFG,data,steps=200,warmup=20)
    benchmark_rows.append(row)
    print(row)
bench={r["model"]:r for r in benchmark_rows}
write_csv(pd.DataFrame(benchmark_rows),DIRS["results"] / "benchmark_results.csv")
CFG,_guard_log=apply_time_guard(bench,CFG,data)
# Recompute and save parameter counts after any approved time-guard changes to hidden size.
actual_param_rows=[]
for name in MODEL_ORDER+("birnn_matched",):
    model=build_model(name,CFG,vocab.V).to(device)
    actual=count_params(model)
    H=hidden_size_for(name,CFG,vocab.V)
    D=2 if name in ("birnn","birnn_matched") else 1
    gates=1 if name in ("simple_rnn","birnn","birnn_matched") else (4 if name=="lstm" else 3)
    expected=expected_params(gates,D,vocab.V,CFG.model.embed_dim,H)
    assert actual==expected, f"Final config parameter count mismatch for {name}: {actual} vs {expected}"
    actual_param_rows.append({"name":name,"params":actual,"expected":expected,"hidden_size":H,"directions":D})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
param_table=pd.DataFrame(actual_param_rows)
write_csv(param_table,DIRS["results"] / "parameter_counts.csv")
RUN["parameter_counts"]={r["name"]:int(r["params"]) for r in actual_param_rows}
RUN["shared_digest"]=shared_digest(CFG)
RUN["train_stride"]=CFG.data.train_stride
RUN["window_counts_after_stride"]={s:int(len(positions[s])) for s in positions}
write_json({"config":CFG,"cfg_changes":RUN["cfg_changes"],"guard_log":RUN["guard_log"],"benchmark":benchmark_rows,
            "gpu_name":gpu_name,"shared_digest":shared_digest(CFG)},DIRS["results"] / "run_config.json")
print("Configuration used for the final runs:\n",json.dumps(asdict(CFG),indent=2))

Toy metric test passed: summed cross-entropy, top-1, top-5 and word-only counts.
Running fixed-subset overfit sanity checks (4 architectures, 300 optimizer steps each).
{'model': 'simple_rnn', 'n_examples': 512, 'steps': 300, 'initial_loss': 9.07719612121582, 'final_loss': 0.0019508046098053455, 'threshold': 4.5297587411208, 'passed': True}
{'model': 'birnn', 'n_examples': 512, 'steps': 300, 'initial_loss': 9.076519012451172, 'final_loss': 0.0002528009645175189, 'threshold': 4.5297587411208, 'passed': True}
{'model': 'lstm', 'n_examples': 512, 'steps': 300, 'initial_loss': 9.060400009155273, 'final_loss': 0.11607665568590164, 'threshold': 4.5297587411208, 'passed': True}
{'model': 'gru', 'n_examples': 512, 'steps': 300, 'initial_loss': 9.058574676513672, 'final_loss': 0.004301448352634907, 'threshold': 4.5297587411208, 'passed': True}
Benchmarking 4 model types; 20 warmup + 200 timed updates per architecture.
{'model': 'simple_rnn', 'step_s': 0.021979164130000017, 'timed_steps': 200, '

## Section 9 — Train the four main models under shared conditions

In [10]:
@dataclass
class RunResult:
    name:str; seed:int; group:str; params:int; V:int; gpu_name:str
    epochs_run:int; best_epoch:int; best_val_loss:float
    train_time_s:float; time_to_best_s:float; sec_per_epoch:float
    stop_reason:str; best_path:str; last_path:str; shared_digest:str; history:list

TEST_EVALUATED=set()

def _rng_payload():
    return {"rng_python":random.getstate(),"rng_numpy":np.random.get_state(),
            "rng_torch":torch.get_rng_state(),"rng_cuda":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def _restore_rng(payload):
    random.setstate(payload["rng_python"]); np.random.set_state(payload["rng_numpy"]); torch.set_rng_state(payload["rng_torch"])
    if torch.cuda.is_available() and payload.get("rng_cuda"): torch.cuda.set_rng_state_all(payload["rng_cuda"])

def train_one_model(name,cfg,data,seed,group="main",resume=False):
    if name not in MODEL_CLASSES: raise KeyError(name)
    set_seed(seed)
    model=build_model(name,cfg,data.vocab.V).to(device)
    params=count_params(model)
    opt=Adam(model.parameters(),lr=cfg.train.lr,weight_decay=cfg.train.weight_decay)
    sched=ReduceLROnPlateau(opt,mode="min",factor=cfg.train.sched_factor,patience=cfg.train.sched_patience,min_lr=cfg.train.sched_min_lr)
    train_loader=WindowLoader(data.streams["train"],data.positions["train"],cfg.data.L,cfg.train.batch_size,True,seed,device)
    best_path=DIRS["checkpoints"] / f"{name}_seed{seed}_best.pt"
    last_path=DIRS["checkpoints"] / f"{name}_seed{seed}_last.pt"
    history=[]; best_val=float("inf"); best_epoch=0; bad_epochs=0; cum_time=0.0; time_to_best=0.0; start_epoch=0
    if resume and last_path.exists():
        last=load_checkpoint(last_path,full=True)
        if last["shared_digest"] != shared_digest(cfg): raise RuntimeError("Cannot resume: config digest differs from checkpoint.")
        if last["vocab_sha256"] != data.vocab.sha256: raise RuntimeError("Cannot resume: vocabulary differs from checkpoint.")
        model.load_state_dict(last["model_state"]); opt.load_state_dict(last["optimizer_state"]); sched.load_state_dict(last["scheduler_state"])
        history=last["history"]; best_val=last["best_val"]; best_epoch=last["best_epoch"]; bad_epochs=last["bad_epochs"]
        cum_time=last["cum_time"]; time_to_best=last.get("time_to_best_s",0.0); start_epoch=int(last["epoch"])
        _restore_rng(last)
        print(f"Resuming {name} seed {seed} after epoch {start_epoch}.")
    else:
        # Reset dropout RNG after initialisation so each model's training randomness starts from the same seed.
        set_seed(seed)
    stop_reason="max_epochs"
    for epoch_index in range(start_epoch,cfg.train.max_epochs):
        train_loader.set_epoch(epoch_index)
        digest=train_loader.perm_digest()
        lr=float(opt.param_groups[0]["lr"])
        model.train(); _sync_device(); t0=time.perf_counter()
        loss_sum=torch.zeros((),dtype=torch.float64,device=device); n_seen=0
        for x,y in train_loader:
            opt.zero_grad(set_to_none=True)
            logits=model(x)
            loss=F.cross_entropy(logits,y,reduction="mean")
            loss.backward()
            clip_grad_norm_(model.parameters(),cfg.train.clip_norm)
            opt.step()
            loss_sum += loss.detach().double()*y.numel(); n_seen += y.numel()
        _sync_device(); epoch_train_time=time.perf_counter()-t0
        cum_time += epoch_train_time
        train_loss_trainmode=float((loss_sum/max(1,n_seen)).item())
        if not math.isfinite(train_loss_trainmode): raise FloatingPointError(f"{name} epoch {epoch_index+1}: non-finite training loss")
        val_metrics=evaluate(model,data.loaders["val"],data.is_word,data.vocab.UNK,k=cfg.train.topk)
        train_sub_metrics=evaluate(model,data.loaders["train_sub"],data.is_word,data.vocab.UNK,k=cfg.train.topk)
        train_eval_loss=float(train_sub_metrics["loss"])
        sched.step(val_metrics["loss"])
        best_before=best_val
        is_best=val_metrics["loss"] < best_val
        if is_best:
            best_val=val_metrics["loss"]; best_epoch=epoch_index+1; time_to_best=cum_time
            save_checkpoint(best_path,model_state=model.state_dict(),name=name,seed=seed,group=group,epoch=epoch_index+1,
                val_metrics=val_metrics,train_eval_loss_at_best=train_eval_loss,cfg=asdict(cfg),shared_digest=shared_digest(cfg),
                vocab_sha256=data.vocab.sha256,V=data.vocab.V,params=params,gpu_name=gpu_name,torch_version=str(torch.__version__))
        if val_metrics["loss"] < best_before-cfg.train.es_min_delta: bad_epochs=0
        else: bad_epochs += 1
        row={"epoch":epoch_index+1,"lr":lr,"steps":len(train_loader),"train_loss_trainmode":train_loss_trainmode,
             "train_eval_loss":train_eval_loss,"val_loss":val_metrics["loss"],"val_ppl":val_metrics["ppl"],
             "val_top1":val_metrics["top1"],"val_top5":val_metrics["top5"],"epoch_train_time_s":epoch_train_time,
             "cum_train_time_s":cum_time,"perm_digest":digest,"is_best":bool(is_best),"bad_epochs":bad_epochs}
        history.append(row)
        payload={"model_state":model.state_dict(),"name":name,"seed":seed,"group":group,"epoch":epoch_index+1,
                 "val_metrics":val_metrics,"train_eval_loss_at_best":train_eval_loss if is_best else None,
                 "cfg":asdict(cfg),"shared_digest":shared_digest(cfg),"vocab_sha256":data.vocab.sha256,"V":data.vocab.V,
                 "params":params,"gpu_name":gpu_name,"torch_version":str(torch.__version__),"optimizer_state":opt.state_dict(),
                 "scheduler_state":sched.state_dict(),"history":history,"best_val":best_val,"best_epoch":best_epoch,
                 "bad_epochs":bad_epochs,"cum_time":cum_time,"time_to_best_s":time_to_best,**_rng_payload()}
        save_checkpoint(last_path,**payload)
        print(f"{name:14s} seed={seed} epoch={epoch_index+1:02d} train={train_loss_trainmode:.4f} train_eval={train_eval_loss:.4f} val={val_metrics['loss']:.4f} ppl={val_metrics['ppl']:.2f} time={epoch_train_time:.1f}s best={best_epoch}")
        if bad_epochs >= cfg.train.es_patience:
            stop_reason="early_stop"; break
    history_df=pd.DataFrame(history)
    hist_name=f"training_history_{name}.csv" if seed==42 else f"training_history_{name}_seed{seed}.csv"
    if group=="ablation": hist_name=f"training_history_{name}_seed{seed}.csv"
    write_csv(history_df,DIRS["results"]/hist_name)
    if not best_path.exists(): raise RuntimeError(f"No best checkpoint was written for {name}, seed {seed}.")
    epochs_run=len(history)
    result=RunResult(name=name,seed=seed,group=group,params=params,V=data.vocab.V,gpu_name=gpu_name,
        epochs_run=epochs_run,best_epoch=best_epoch,best_val_loss=best_val,train_time_s=cum_time,
        time_to_best_s=time_to_best,sec_per_epoch=cum_time/max(1,epochs_run),stop_reason=stop_reason,
        best_path=str(best_path),last_path=str(last_path),shared_digest=shared_digest(cfg),history=history)
    del model,opt,sched,train_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

# Save data digests to prove the held-out streams were not modified by training.
def tensor_sha256(t): return hashlib.sha256(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()
val_stream_digest_before=tensor_sha256(data.streams["val"])
test_stream_digest_before=tensor_sha256(data.streams["test"])
flags_before=global_flags_snapshot()
training_started=time.time()
main_runs=[]
for name in MODEL_ORDER:
    if global_flags_snapshot()!=flags_before: raise RuntimeError("Determinism/TF32 flags changed during execution.")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    main_runs.append(train_one_model(name,CFG,data,CFG.run.model_seed,group="main",resume=False))
seed42_wall_seconds=time.time()-training_started
RUN["train_seed42_wall_s"]=seed42_wall_seconds
RUN["seeds_run"]=[CFG.run.model_seed]

# Optional extra seeds: only when seed 42 took under two hours AND the estimated total remains within the 6h run budget.
extra_estimate_hours=(2*seed42_wall_seconds + CFG.run.gen_reserve_h*3600)/3600
elapsed_before_extra=(time.time()-RUN["started_unix"])/3600
if seed42_wall_seconds < CFG.run.extra_seed_gate_h*3600 and elapsed_before_extra+extra_estimate_hours <= CFG.run.time_budget_h:
    extra_seed_runs=[]
    for extra_seed in CFG.run.extra_seeds:
        for name in MODEL_ORDER:
            extra_seed_runs.append(train_one_model(name,CFG,data,extra_seed,group="main",resume=False))
    main_runs.extend(extra_seed_runs)
    RUN["seeds_run"].extend(list(CFG.run.extra_seeds))
    RUN["extra_seeds_decision"]="ran seeds 43 and 44: seed-42 training <2h and estimated total within budget"
else:
    RUN["extra_seeds_decision"]=(f"skipped: seed-42 training took {seed42_wall_seconds/3600:.2f}h and/or estimated total would exceed budget")
print("Extra-seed decision:",RUN["extra_seeds_decision"])

# Optional parameter-matched BiRNN control is separate from the four-model winner decision.
optional_runs=[]
elapsed_hours=(time.time()-RUN["started_unix"])/3600
birnn_main=next(r for r in main_runs if r.name=="birnn" and r.seed==CFG.run.model_seed)
birnn_bench=bench["birnn"]
steps_per_epoch=math.ceil(len(data.positions["train"])/CFG.train.batch_size)
est_ablation_epoch=steps_per_epoch*birnn_bench["step_s"] + birnn_bench["val_batch_s"]*(len(data.loaders["val"])+len(data.loaders["train_sub"]))
est_ablation_hours=est_ablation_epoch*birnn_main.epochs_run/3600
remaining_hours=CFG.run.time_budget_h-elapsed_hours
if remaining_hours >= CFG.run.ablation_margin*est_ablation_hours:
    print(f"Running optional parameter-matched BiRNN: estimated {est_ablation_hours:.2f}h, remaining budget {remaining_hours:.2f}h.")
    optional_runs.append(train_one_model("birnn_matched",CFG,data,CFG.run.model_seed,group="ablation",resume=False))
    RUN["optional_ablation_trained"]=True
else:
    print(f"Skipping optional BiRNN capacity control: estimated {est_ablation_hours:.2f}h, remaining budget {remaining_hours:.2f}h.")
    RUN["optional_ablation_trained"]=False

all_runs=main_runs+optional_runs
assert len({r.gpu_name for r in all_runs})==1, "Models must be trained on the same device/hardware session."
main_seed42=[r for r in main_runs if r.seed==CFG.run.model_seed]
assert [r.name for r in main_seed42]==list(MODEL_ORDER)
for r in main_seed42: assert r.shared_digest==shared_digest(CFG)
# Common epochs must use an identical data-order digest across the four main models.
common_epochs=min(r.epochs_run for r in main_seed42)
for ep in range(common_epochs):
    digests={r.history[ep]["perm_digest"] for r in main_seed42}
    assert len(digests)==1, f"Batch order differs across models at epoch {ep+1}: {digests}"
assert tensor_sha256(data.streams["val"])==val_stream_digest_before
assert tensor_sha256(data.streams["test"])==test_stream_digest_before
RUN["all_models_gpu_names"]={r.name+"_seed"+str(r.seed):r.gpu_name for r in all_runs}
RUN["training_runs"]=[{**{k:v for k,v in asdict(r).items() if k!="history"},"history_rows":len(r.history)} for r in all_runs]
print(f"Completed seed-42 four-model run in {seed42_wall_seconds/60:.1f} min; common epochs checked: {common_epochs}.")

simple_rnn     seed=42 epoch=01 train=5.1713 train_eval=4.5456 val=4.6100 ppl=100.48 time=64.5s best=1
simple_rnn     seed=42 epoch=02 train=4.8069 train_eval=4.3458 val=4.5182 ppl=91.67 time=64.1s best=2
simple_rnn     seed=42 epoch=03 train=4.6834 train_eval=4.2070 val=4.4640 ppl=86.83 time=64.5s best=3
simple_rnn     seed=42 epoch=04 train=4.6107 train_eval=4.1203 val=4.4566 ppl=86.19 time=65.0s best=4
simple_rnn     seed=42 epoch=05 train=4.5602 train_eval=4.0548 val=4.4549 ppl=86.05 time=64.5s best=5
simple_rnn     seed=42 epoch=06 train=4.5203 train_eval=4.0080 val=4.4548 ppl=86.04 time=64.2s best=6
simple_rnn     seed=42 epoch=07 train=4.4907 train_eval=3.9626 val=4.4571 ppl=86.23 time=64.1s best=6
simple_rnn     seed=42 epoch=08 train=4.2979 train_eval=3.7747 val=4.3589 ppl=78.17 time=64.3s best=8
simple_rnn     seed=42 epoch=09 train=4.2237 train_eval=3.7282 val=4.3462 ppl=77.18 time=64.5s best=9
simple_rnn     seed=42 epoch=10 train=4.1831 train_eval=3.6887 val=4.3462 ppl=77.

## Section 10 — Best-checkpoint validation/test evaluation and baselines

In [11]:
def _load_best_model(run:RunResult,cfg:Config,data:DataBundle):
    ckpt=load_checkpoint(run.best_path,full=False)
    if ckpt["vocab_sha256"] != data.vocab.sha256: raise RuntimeError(f"{run.name}: checkpoint vocabulary hash mismatch.")
    if ckpt["shared_digest"] != shared_digest(cfg): raise RuntimeError(f"{run.name}: checkpoint configuration digest mismatch.")
    model=build_model(run.name,cfg,data.vocab.V).to(device)
    model.load_state_dict(ckpt["model_state"]); model.eval()
    return model,ckpt

def final_evaluate(run:RunResult,cfg:Config,data:DataBundle):
    key=(run.name,run.seed)
    if key in TEST_EVALUATED: raise RuntimeError(f"Test set has already been evaluated for {key}; refusing a second test evaluation.")
    model,ckpt=_load_best_model(run,cfg,data)
    val=evaluate(model,data.loaders["val"],data.is_word,data.vocab.UNK,k=cfg.train.topk)
    if abs(val["loss"]-float(ckpt["val_metrics"]["loss"])) > 1e-4:
        raise RuntimeError(f"{run.name}: reloaded checkpoint validation loss {val['loss']:.6f} differs from saved best {ckpt['val_metrics']['loss']:.6f}.")
    test=evaluate(model,data.loaders["test"],data.is_word,data.vocab.UNK,k=cfg.train.topk)
    TEST_EVALUATED.add(key)
    train_eval=float(ckpt["train_eval_loss_at_best"])
    row={"model":run.name,"group":run.group,"params":run.params,"V":run.V,"seed":run.seed,"gpu_name":run.gpu_name,
         "epochs_run":run.epochs_run,"best_epoch":run.best_epoch,"N_val":val["N"],"N_test":test["N"],
         "val_loss":val["loss"],"val_ppl":val["ppl"],"val_top1":val["top1"],"val_top5":val["top5"],
         "test_loss":test["loss"],"test_ppl":test["ppl"],"test_top1":test["top1"],"test_top5":test["top5"],
         "test_top1_words_only":test["top1_words_only"],"test_unk_target_rate":test["unk_target_rate"],
         "train_eval_loss_at_best":train_eval,"gap":val["loss"]-train_eval,"train_time_s":run.train_time_s,
         "sec_per_epoch":run.sec_per_epoch,"time_to_best_s":run.time_to_best_s}
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

metrics_rows=[]
for run in all_runs:
    metrics_rows.append(final_evaluate(run,CFG,data))
metrics_summary=pd.DataFrame(metrics_rows)
assert metrics_summary[metrics_summary.group=="main"]["N_test"].nunique()==1
assert metrics_summary[metrics_summary.group=="main"]["N_val"].nunique()==1

# Unigram baseline from training-target counts; no smoothing, as required by the plan.
train_counts_cpu=data.train_counts.detach().cpu().numpy().astype(np.float64)
total_train_targets=float(train_counts_cpu.sum())
unigram_probs=train_counts_cpu/max(1.0,total_train_targets)

def baseline_metrics(split:str):
    target_pos=data.positions_full[split]
    target_ids=data.streams[split][target_pos].detach().cpu().numpy().astype(np.int64)
    probs=unigram_probs[target_ids]
    if np.any(probs<=0):
        unseen=np.unique(target_ids[probs<=0]).tolist()
        raise RuntimeError(f"Unigram baseline has zero train probability for {split} target ids {unseen[:20]}; the plan requires inspection, not silent smoothing.")
    loss=float(-np.log(probs).mean())
    return {"baseline":"unigram","split":split,"N":int(len(target_ids)),"loss":loss,"perplexity":float(math.exp(min(loss,80.0)))}

baseline_rows=[
    {"baseline":"uniform","split":s,"N":int(len(data.positions_full[s])),"loss":math.log(vocab.V),"perplexity":float(vocab.V)}
    for s in ("val","test")
]
for s in ("val","test"): baseline_rows.append(baseline_metrics(s))
baselines_df=pd.DataFrame(baseline_rows)
write_csv(baselines_df,DIRS["results"] / "baselines.csv")

val_uni=float(baselines_df[(baselines_df.baseline=="unigram")&(baselines_df.split=="val")]["perplexity"].iloc[0])
for _,r in metrics_summary[metrics_summary.group=="main"].iterrows():
    if float(r["val_ppl"]) >= val_uni:
        msg=f"{r['model']} seed {r['seed']} validation perplexity {r['val_ppl']:.3f} did not beat training unigram baseline {val_uni:.3f}; inspect the pipeline before interpreting the ranking."
        RUN["warnings"].append(msg); print("WARNING:",msg)
    else:
        print(f"Baseline gate: {r['model']} seed {r['seed']} val PPL {r['val_ppl']:.3f} < unigram {val_uni:.3f}")
RUN["unigram_validation_ppl"]=val_uni
RUN["unigram_gate_passed"]=bool((metrics_summary[metrics_summary.group=="main"]["val_ppl"]<val_uni).all())

# The main comparison uses equal-width models. If extra seeds ran, add mean and std summary rows per architecture.
aggregate_rows=[]
main_only=metrics_summary[metrics_summary.group=="main"].copy()
if main_only["seed"].nunique()>1:
    numeric_cols=["params","V","epochs_run","best_epoch","N_val","N_test","val_loss","val_ppl","val_top1","val_top5",
                  "test_loss","test_ppl","test_top1","test_top5","test_top1_words_only","test_unk_target_rate",
                  "train_eval_loss_at_best","gap","train_time_s","sec_per_epoch","time_to_best_s"]
    for name,grp in main_only.groupby("model"):
        for stat in ("mean","std"):
            vals=grp[numeric_cols].mean(numeric_only=True) if stat=="mean" else grp[numeric_cols].std(numeric_only=True,ddof=1).fillna(0.0)
            row={"model":name,"group":"main","seed":stat,"gpu_name":gpu_name}
            row.update(vals.to_dict()); aggregate_rows.append(row)
if aggregate_rows:
    metrics_summary=pd.concat([metrics_summary,pd.DataFrame(aggregate_rows)],ignore_index=True)

expected_cols=["model","group","params","V","seed","gpu_name","epochs_run","best_epoch","N_val","N_test","val_loss","val_ppl","val_top1","val_top5",
               "test_loss","test_ppl","test_top1","test_top5","test_top1_words_only","test_unk_target_rate","train_eval_loss_at_best","gap","train_time_s","sec_per_epoch","time_to_best_s"]
metrics_summary=metrics_summary.reindex(columns=expected_cols)
write_csv(metrics_summary,DIRS["results"] / "metrics_summary.csv")
write_md_table(metrics_summary,DIRS["results"] / "metrics_summary.md",{"val_loss":4,"val_ppl":2,"test_loss":4,"test_ppl":2,"val_top1":4,"test_top1":4,"train_time_s":1,"sec_per_epoch":1})
print("\nMain test results (seed 42):")
print(metrics_summary[(metrics_summary.group=="main")&(metrics_summary.seed==42)][["model","params","best_epoch","val_ppl","test_ppl","test_top1","test_top1_words_only","train_time_s"]].to_string(index=False))

Baseline gate: simple_rnn seed 42 val PPL 72.379 < unigram 349.931
Baseline gate: birnn seed 42 val PPL 95.140 < unigram 349.931
Baseline gate: lstm seed 42 val PPL 57.383 < unigram 349.931
Baseline gate: gru seed 42 val PPL 62.341 < unigram 349.931

Main test results (seed 42):
     model   params  best_epoch   val_ppl   test_ppl  test_top1  test_top1_words_only  train_time_s
simple_rnn  7532952          20 72.378717  79.135110   0.239089              0.161535   1285.993919
     birnn 13379992          20 95.140369 104.892233   0.235255              0.157637   2962.580282
      lstm 10291608           6 57.382926  63.540011   0.262265              0.174436   1744.486036
       gru  9372056           7 62.340702  68.622219   0.250618              0.172756   1386.409853


## Section 11 — Training plots

In [12]:
def plot_loss_curves(run_list,path):
    fig,axes=plt.subplots(1,2,figsize=(13,4.5))
    for run in run_list:
        if run.seed!=CFG.run.model_seed or run.group!="main": continue
        hist=pd.DataFrame(run.history)
        label=MODEL_LABELS.get(run.name,run.name); colour=MODEL_COLOURS.get(run.name)
        axes[0].plot(hist["epoch"],hist["val_loss"],label=label,color=colour)
        axes[1].plot(hist["epoch"],hist["train_eval_loss"],label=label,color=colour)
    axes[0].set_title("Validation loss by epoch"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Mean token NLL (nats)")
    axes[1].set_title("Train-subset evaluation loss by epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Mean token NLL (nats)")
    for ax in axes: ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)

def plot_val_ppl(run_list,path):
    fig,ax=plt.subplots(figsize=(7,4.5))
    for run in run_list:
        if run.seed!=CFG.run.model_seed or run.group!="main": continue
        hist=pd.DataFrame(run.history)
        ax.plot(hist["epoch"],hist["val_ppl"],label=MODEL_LABELS.get(run.name,run.name),color=MODEL_COLOURS.get(run.name))
    ax.set_title("Validation perplexity by epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("Perplexity"); ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)

def plot_test_bars(summary_df,path):
    df=summary_df[(summary_df.group=="main")&(summary_df.seed==42)].copy()
    metrics=[("test_ppl","Test perplexity"),("test_top1","Test top-1 accuracy"),("train_time_s","Training time (s)"),("params","Trainable parameters")]
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for ax,(col,title) in zip(axes.flatten(),metrics):
        vals=df[col].astype(float).values
        ax.bar([MODEL_LABELS.get(x,x) for x in df.model],vals,color=[MODEL_COLOURS.get(x,"gray") for x in df.model])
        ax.set_title(title); ax.set_ylabel(title); ax.tick_params(axis="x",rotation=18); ax.grid(axis="y",alpha=.2)
    fig.tight_layout(); fig.savefig(path,dpi=150); plt.close(fig)

plot_loss_curves(all_runs,DIRS["plots"] / "loss_curves.png")
plot_val_ppl(all_runs,DIRS["plots"] / "val_perplexity_curves.png")
plot_test_bars(metrics_summary,DIRS["plots"] / "test_metrics_bars.png")
print("Saved:",*[p.name for p in sorted(DIRS["plots"].glob("*.png"))],sep="\n - ")

Saved:
 - corpus_eda.png
 - loss_curves.png
 - test_metrics_bars.png
 - val_perplexity_curves.png


## Section 12 — Generation setup, vocabulary provenance and seed checks

In [13]:
STOPWORDS=set("a an the and or but of to in on at by for with from into upon as is was were be been are am i you he she it we they me him her us them my your his its our their this that these those had have has not no so if then there when what who which".split())
known_word_tokens=[t for t in vocab.itos[3:] if WORD_RE.fullmatch(t)]
manual_replacements=dict(CFG.gen.seed_replacements)

def _format_seed_tokens(tokens):
    s=" ".join(tokens)
    s=re.sub(r"\s+([.,!?;:])",r"\1",s)
    s=s.replace('" ', '"').replace(' "','"')
    return s.strip()

def resolve_seed(seed_text:str):
    toks=tokenize(seed_text); changed=[]
    for i,tok in enumerate(list(toks)):
        if tok in vocab.stoi: continue
        if not WORD_RE.fullmatch(tok):
            raise RuntimeError(f"Seed punctuation/token {tok!r} is missing from vocabulary; inspect tokenizer/vocabulary.")
        replacement=manual_replacements.get(tok)
        if replacement is None:
            matches=difflib.get_close_matches(tok,known_word_tokens,n=1,cutoff=0.48)
            if matches: replacement=matches[0]
            else:
                same_prefix=[w for w in known_word_tokens if w.startswith(tok[:1])]
                replacement=same_prefix[0] if same_prefix else known_word_tokens[0]
        if replacement not in vocab.stoi or replacement in {"<unk>","<bos>","<eos>"}:
            raise RuntimeError(f"Invalid seed replacement {tok!r} -> {replacement!r}.")
        toks[i]=replacement; changed.append((tok,replacement))
    final_text=_format_seed_tokens(toks)
    ids=encode(final_text,vocab,add_eos=False)
    if vocab.UNK in ids or vocab.BOS in ids: raise AssertionError(f"Seed still has UNK/BOS after replacement: {final_text}")
    return final_text,ids,changed

def longest_run(seed_ids:list[int],stream_ids:Sequence[int])->int:
    if not seed_ids or len(stream_ids)==0: return 0
    # O(len(stream)*len(seed)) dynamic programming without per-token dict allocation.
    m=len(seed_ids); dp=[0]*m; best=0
    for raw in stream_ids:
        x=int(raw)
        for j in range(m-1,0,-1):
            dp[j]=dp[j-1]+1 if x==int(seed_ids[j]) else 0
            if dp[j]>best: best=dp[j]
        dp[0]=1 if x==int(seed_ids[0]) else 0
        if dp[0]>best: best=dp[0]
        if best==m: return best
    return int(best)

seed_ids={}; resolved_seed_texts={}; seed_replacement_notes={}
seed_rows=[]
for seed_id,seed_text in CFG.gen.seed_texts:
    resolved,ids,replacements=resolve_seed(seed_text)
    if len(ids)>=CFG.data.L: raise AssertionError(f"{seed_id} must be shorter than context length {CFG.data.L}.")
    seed_ids[seed_id]=ids; resolved_seed_texts[seed_id]=resolved; seed_replacement_notes[seed_id]=replacements
    row={"seed_id":seed_id,"original_seed_text":seed_text,"seed_text":resolved,"n_tokens":len(ids),
         "replacement_note":"; ".join(f"{a}->{b}" for a,b in replacements) or "none"}
    for split in ("train","val","test"):
        matches=[longest_run(ids,ids_by_work[w]) for w in names_by_split[split]]
        row[f"longest_run_{split}"]=max(matches,default=0)
    row["longest_run_overall"]=max(row[f"longest_run_{s}"] for s in ("train","val","test"))
    seed_rows.append(row)
seed_check_df=pd.DataFrame(seed_rows)
write_csv(seed_check_df,DIRS["results"] / "seed_check.csv")
print(seed_check_df[["seed_id","seed_text","n_tokens","longest_run_train","longest_run_val","longest_run_test","replacement_note"]].to_string(index=False))

# Load best checkpoints for the seed-42 generation matrix; never rebuild the vocabulary after training.
gen_models={}
for run in all_runs:
    if run.seed != CFG.run.model_seed: continue
    model,ckpt=_load_best_model(run,CFG,data)
    if ckpt["vocab_sha256"] != vocab.sha256: raise RuntimeError(f"{run.name}: generation vocabulary/checkpoint hash mismatch")
    model.eval(); assert not model.training
    gen_models[run.name]=model
assert all(name in gen_models for name in MODEL_ORDER)

seed_id                                            seed_text  n_tokens  longest_run_train  longest_run_val  longest_run_test replacement_note
     S1      it was a cold foggy morning in baker street and        10                  4                3                 3             none
     S2                       "my dear watson,"said holmes,"        10                  8                7                 7             none
     S3      from the mud upon your boots i can see that you        11                  5                5                 4             none
     S4              the inspector knelt beside the body and         7                  3                2                 2             none
     S5   a lady in a thick veil was shown into the room and        12                  4                5                 4             none
     S6 i have seldom seen my friend so moved as he was when        12                  4                3                 2             none


## Section 13 — Full autoregressive generation matrix

In [14]:
@torch.no_grad()
def generate(model,seed_token_ids:list[int],temperature:float,n_new:int,rng:Optional[torch.Generator],L:int,vocab:Vocab,device):
    model.eval()
    seed_tensor=torch.tensor(seed_token_ids,dtype=torch.long,device=device)
    if seed_tensor.numel()<L:
        ctx=torch.cat([torch.full((L-seed_tensor.numel(),),vocab.BOS,dtype=torch.long,device=device),seed_tensor])
    else:
        ctx=seed_tensor[-L:]
    out=[]
    for _ in range(n_new):
        logits=model(ctx.unsqueeze(0))[0].float().clone()
        logits[vocab.UNK]=-torch.inf; logits[vocab.BOS]=-torch.inf
        if temperature==0:
            nxt=int(torch.argmax(logits).item())
        else:
            if temperature<=0 or not math.isfinite(temperature): raise ValueError("Temperature must be zero (greedy) or a positive finite value.")
            probs=torch.softmax(logits/float(temperature),dim=-1).detach().cpu()
            if not torch.isfinite(probs).all() or float(probs.sum().item())<=0:
                raise FloatingPointError("Invalid probability distribution during temperature sampling.")
            probs=probs/probs.sum()
            if rng is None: raise ValueError("A CPU torch.Generator is required for sampled generation.")
            nxt=int(torch.multinomial(probs,1,generator=rng).item())
        if nxt in (vocab.UNK,vocab.BOS): raise AssertionError("Generation emitted a forbidden special token.")
        out.append(nxt)
        ctx=torch.cat([ctx[1:],torch.tensor([nxt],dtype=torch.long,device=device)])
    return out

def detokenize_generation(ids:Sequence[int],vocab:Vocab)->str:
    pieces=[]
    for raw_id in ids:
        tok=vocab.itos[int(raw_id)]
        if tok=="<eos>":
            pieces.append("\n")
        elif tok in {".",",","!","?",";",":"}:
            if pieces and pieces[-1] not in {" ","\n"}: pieces.append(tok)
            else: pieces.append(tok)
            pieces.append(" ")
        elif tok=='"':
            # Lightweight quote spacing; readability is secondary to preserving token order.
            if pieces and pieces[-1] not in {" ","\n"}: pieces.append(" ")
            pieces.append(tok)
        else:
            if pieces and pieces[-1] not in {" ","\n",'"'}: pieces.append(" ")
            pieces.append(tok)
    text="".join(pieces)
    text=re.sub(r"[ \t]+\n","\n",text)
    text=re.sub(r"\n{3,}","\n\n",text)
    text=re.sub(r"\s+([.,!?;:])",r"\1",text)
    return text.strip()

def run_generation_matrix(models,seeds,cfg,vocab):
    rows=[]
    for model_name in list(MODEL_ORDER)+(["birnn_matched"] if "birnn_matched" in models else []):
        model=models[model_name]
        assert not model.training, f"{model_name} must be in eval mode before generation."
        group="ablation" if model_name=="birnn_matched" else "main"
        for seed_idx,(seed_id,seed_text) in enumerate(cfg.gen.seed_texts,start=1):
            resolved_text=resolved_seed_texts[seed_id]
            ids=seeds[seed_id]
            # Sampled conditions: 5 independent samples at each temperature.
            for temp_idx,temp in enumerate(cfg.gen.temperatures):
                for sample_idx in range(cfg.gen.samples_per_cell):
                    rng_seed=cfg.gen.rng_base+seed_idx*1000+temp_idx*100+sample_idx
                    rng=torch.Generator(device="cpu").manual_seed(rng_seed)
                    new_ids=generate(model,ids,float(temp),cfg.gen.n_new,rng,cfg.data.L,vocab,device)
                    rows.append({"model":model_name,"group":group,"seed_id":seed_id,"seed_text":resolved_text,
                        "temperature":float(temp),"condition":f"T{temp:.1f}","sample_index":sample_idx,"rng_seed":rng_seed,
                        "ids":new_ids,"tokens":[vocab.itos[i] for i in new_ids],"text":detokenize_generation(new_ids,vocab)})
            if cfg.gen.include_greedy:
                new_ids=generate(model,ids,0.0,cfg.gen.n_new,None,cfg.data.L,vocab,device)
                rows.append({"model":model_name,"group":group,"seed_id":seed_id,"seed_text":resolved_text,
                    "temperature":0.0,"condition":"greedy","sample_index":0,"rng_seed":None,
                    "ids":new_ids,"tokens":[vocab.itos[i] for i in new_ids],"text":detokenize_generation(new_ids,vocab)})
    return rows

rows=run_generation_matrix(gen_models,seed_ids,CFG,vocab)
expected_main=4*len(seed_ids)*len(CFG.gen.temperatures)*CFG.gen.samples_per_cell + 4*len(seed_ids)*int(CFG.gen.include_greedy)
expected_ablation=0
if "birnn_matched" in gen_models: expected_ablation=len(seed_ids)*len(CFG.gen.temperatures)*CFG.gen.samples_per_cell+len(seed_ids)*int(CFG.gen.include_greedy)
assert len(rows)==expected_main+expected_ablation, f"Generation row count {len(rows)} expected {expected_main+expected_ablation}"
assert all(len(r["ids"])==CFG.gen.n_new for r in rows)
assert all(vocab.UNK not in r["ids"] and vocab.BOS not in r["ids"] for r in rows)
# Same random seed for the corresponding condition across the four main models.
for seed_id in seed_ids:
    for temp in CFG.gen.temperatures:
        for sample_idx in range(CFG.gen.samples_per_cell):
            found={r["rng_seed"] for r in rows if r["group"]=="main" and r["seed_id"]==seed_id and r["temperature"]==temp and r["sample_index"]==sample_idx}
            assert len(found)==1, f"RNG seed mismatch across models for {seed_id}, T={temp}, sample={sample_idx}: {found}"
# Same-session reproducibility check: LSTM, S1, T=1.0, sample 0.
check_row=next(r for r in rows if r["model"]=="lstm" and r["seed_id"]=="S1" and r["condition"]=="T1.0" and r["sample_index"]==0)
rng_check=torch.Generator(device="cpu").manual_seed(check_row["rng_seed"])
redo=generate(gen_models["lstm"],seed_ids["S1"],1.0,CFG.gen.n_new,rng_check,CFG.data.L,vocab,device)
assert redo==check_row["ids"], "Generation is not reproducible for the reference cell in this session."

# Keep structured JSONL for reproducibility and CSV/Markdown for inspection.
jsonl_path=DIRS["samples"] / "generation_samples.jsonl"
with jsonl_path.open("w",encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(_jsonable(r),ensure_ascii=False)+"\n")
rows_df=pd.DataFrame(rows)
write_csv(rows_df,DIRS["samples"] / "generation_samples.csv")
md_lines=["# Generated text samples", "", "Each sample contains exactly 50 newly generated tokens; seed text is shown separately.", ""]
for seed_id in seed_ids:
    md_lines += [f"## {seed_id}: {resolved_seed_texts[seed_id]}",""]
    for cond in ["T0.5","T1.0","T1.5","greedy"]:
        md_lines += [f"### {cond}",""]
        for model_name in list(MODEL_ORDER)+(["birnn_matched"] if "birnn_matched" in gen_models else []):
            rr=next(r for r in rows if r["seed_id"]==seed_id and r["condition"]==cond and r["model"]==model_name and r["sample_index"]==0)
            md_lines += [f"**{MODEL_LABELS.get(model_name,model_name)}** — {rr['text'].replace(chr(10),' / ')}",""]
(DIRS["samples"] / "generation_samples.md").write_text("\n".join(md_lines),encoding="utf-8")
print(f"Saved {len(rows)} generated samples. Main expected count: {expected_main}; optional ablation rows: {expected_ablation}.")

Saved 480 generated samples. Main expected count: 384; optional ablation rows: 96.


## Section 14 — Quantitative text-quality metrics and temperature effects

In [16]:
STOPWORDS = set(
    "a an the and or but of to in on at by for with from into upon as "
    "is was were be been are am i you he she it we they me him her us "
    "them my your his its our their this that these those had have has "
    "not no so if then there when what who which".split()
)


def train_ngram_arrays(ids_train_per_work: Iterable[Sequence[int]], V: int):
    if V**3 >= 2**62:
        raise OverflowError(f"V^3={V**3} too large for signed int64 trigram encoding.")

    bigram_chunks = []
    trigram_chunks = []

    for ids in ids_train_per_work:
        arr = np.asarray(ids, dtype=np.int64)

        if len(arr) >= 2:
            bigram_chunks.append(arr[:-1] * V + arr[1:])

        if len(arr) >= 3:
            trigram_chunks.append(
                (arr[:-2] * V + arr[1:-1]) * V + arr[2:]
            )

    bigrams = (
        np.unique(np.concatenate(bigram_chunks))
        if bigram_chunks else np.array([], dtype=np.int64)
    )
    trigrams = (
        np.unique(np.concatenate(trigram_chunks))
        if trigram_chunks else np.array([], dtype=np.int64)
    )

    return np.sort(bigrams), np.sort(trigrams)


train_ids_per_work = [
    ids_by_work[w] for w in names_by_split["train"]
]

train_bigram_codes, train_trigram_codes = train_ngram_arrays(
    train_ids_per_work, vocab.V
)


def _ngram_codes(ids, n, V):
    a = np.asarray(ids, dtype=np.int64)

    if len(a) < n:
        return np.array([], dtype=np.int64)

    if n == 2:
        return a[:-1] * V + a[1:]

    if n == 3:
        return (a[:-2] * V + a[1:-1]) * V + a[2:]

    if n == 4:
        return ((a[:-3] * V + a[1:-2]) * V + a[2:-1]) * V + a[3:]

    raise ValueError("Only n=2,3,4 supported.")


def cell_metrics(cell_rows, bigrams, trigrams, vocab: Vocab, V: int, scope: str):
    if scope == "sample0":
        cell_rows = [r for r in cell_rows if r["sample_index"] == 0]

    if not cell_rows:
        raise ValueError("Empty generation metric cell.")

    per_sample = []
    all_uni = []
    all_bigrams = []
    valid_b = []
    valid_t = []
    adj_same = 0
    adj_total = 0
    carry = []
    quotes = []

    for r in cell_rows:
        ids = list(map(int, r["ids"]))

        if any(i < 0 or i >= V for i in ids):
            raise ValueError("Generated token ID is outside the vocabulary.")

        all_uni.extend(ids)

        bg = _ngram_codes(ids, 2, V)
        tg = _ngram_codes(ids, 3, V)
        qg = _ngram_codes(ids, 4, V)

        all_bigrams.extend(bg.tolist())
        valid_b.extend(np.isin(bg, bigrams).tolist())
        valid_t.extend(np.isin(tg, trigrams).tolist())

        rep3 = 1.0 - len(np.unique(tg)) / max(1, len(tg))
        loop = bool(len(qg) > 0 and len(np.unique(qg)) < len(qg))

        per_sample.append({
            "rep3": rep3,
            "loop_rate": float(loop)
        })

        if len(ids) > 1:
            adj_same += sum(
                ids[i] == ids[i - 1] for i in range(1, len(ids))
            )
            adj_total += len(ids) - 1

        seed_toks = tokenize(r["seed_text"])
        content = [
            t for t in seed_toks
            if re.fullmatch(r"[a-z]+(?:'[a-z]+)*", t)
            and t not in STOPWORDS
        ]

        gen_set = {vocab.itos[i] for i in ids}

        carry.append(
            sum(t in gen_set for t in set(content))
            / max(1, len(set(content)))
        )

        all_text_toks = seed_toks + [vocab.itos[i] for i in ids]
        quotes.append(
            float(sum(t == '"' for t in all_text_toks) % 2 == 0)
        )

    n_samples = len(cell_rows)
    total_uni = len(all_uni)
    total_bg = sum(max(0, len(r["ids"]) - 1) for r in cell_rows)

    pooled_bg = np.asarray(all_bigrams, dtype=np.int64)

    distinct1 = len(set(all_uni)) / max(1, total_uni)
    distinct2 = len(np.unique(pooled_bg)) / max(1, total_bg)

    return {
        "n_samples": n_samples,
        "n_tokens": total_uni,
        "distinct1": distinct1,
        "distinct2": distinct2,
        "rep3": float(np.mean([p["rep3"] for p in per_sample])),
        "loop_rate": float(np.mean([p["loop_rate"] for p in per_sample])),
        "adjacent_repeat": adj_same / max(1, adj_total),
        "bigram_valid": float(np.mean(valid_b)) if valid_b else 0.0,
        "trigram_valid": float(np.mean(valid_t)) if valid_t else 0.0,
        "seed_carryover": float(np.mean(carry)),
        "quote_closure": float(np.mean(quotes)),
    }


metric_rows = []

metric_model_names = list(MODEL_ORDER)
if "birnn_matched" in gen_models:
    metric_model_names.append("birnn_matched")

for model_name in metric_model_names:
    group = "ablation" if model_name == "birnn_matched" else "main"

    for condition in ["T0.5", "T1.0", "T1.5", "greedy"]:
        cell = [
            r for r in rows
            if r["model"] == model_name and r["condition"] == condition
        ]

        for scope in ("pooled", "sample0"):
            met = cell_metrics(
                cell,
                train_bigram_codes,
                train_trigram_codes,
                vocab,
                vocab.V,
                scope
            )

            metric_rows.append({
                "model": model_name,
                "group": group,
                "condition": condition,
                "temperature": (
                    0.0 if condition == "greedy"
                    else float(condition[1:])
                ),
                "scope": scope,
                **met
            })


metrics_df = pd.DataFrame(metric_rows)

write_csv(
    metrics_df,
    DIRS["results"] / "generation_metrics.csv"
)

assert set(metrics_df["condition"]) == {
    "T0.5", "T1.0", "T1.5", "greedy"
}

frac_cols = [
    "distinct1", "distinct2", "rep3", "loop_rate",
    "adjacent_repeat", "bigram_valid", "trigram_valid",
    "seed_carryover", "quote_closure"
]

for c in frac_cols:
    assert metrics_df[c].between(0, 1).all(), (
        f"Metric {c} outside [0,1]."
    )

for name in MODEL_ORDER:
    for cond in ("T0.5", "T1.0", "T1.5"):
        pool = metrics_df[
            (metrics_df.model == name)
            & (metrics_df.condition == cond)
            & (metrics_df.scope == "pooled")
        ].iloc[0]

        s0 = metrics_df[
            (metrics_df.model == name)
            & (metrics_df.condition == cond)
            & (metrics_df.scope == "sample0")
        ].iloc[0]

        assert pool["n_samples"] == 30
        assert s0["n_samples"] == 6
        assert pool["n_tokens"] == 1500
        assert s0["n_tokens"] == 300


# Hand-check of token n-gram encoding.
toy = [1, 2, 1, 2, 1]

toy_bg = _ngram_codes(toy, 2, 10)
toy_tg = _ngram_codes(toy, 3, 10)
toy_qg = _ngram_codes(toy, 4, 10)

assert len(toy_bg) == 4 and len(np.unique(toy_bg)) == 2
assert len(toy_tg) == 3 and len(np.unique(toy_tg)) == 2

# FIX: the two 4-grams are [1,2,1,2] and [2,1,2,1].
assert len(toy_qg) == 2 and len(np.unique(toy_qg)) == 2


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

plot_metrics = [
    ("distinct2", "Distinct-2 (diversity)"),
    ("rep3", "Repeated trigram rate"),
    ("bigram_valid", "Train bigram validity")
]

for ax, (metric, title) in zip(axes, plot_metrics):
    for model_name in MODEL_ORDER:
        ys = []

        for cond in ("T0.5", "T1.0", "T1.5"):
            row = metrics_df[
                (metrics_df.model == model_name)
                & (metrics_df.condition == cond)
                & (metrics_df.scope == "pooled")
            ]
            ys.append(float(row[metric].iloc[0]))

        ax.plot(
            [0.5, 1.0, 1.5],
            ys,
            marker="o",
            label=MODEL_LABELS[model_name],
            color=MODEL_COLOURS[model_name]
        )

        greedy_row = metrics_df[
            (metrics_df.model == model_name)
            & (metrics_df.condition == "greedy")
            & (metrics_df.scope == "sample0")
        ]

        ax.scatter(
            [0.0],
            [float(greedy_row[metric].iloc[0])],
            marker="x",
            color=MODEL_COLOURS[model_name],
            s=60
        )

    ax.set_title(title)
    ax.set_xlabel("Sampling temperature (greedy reference at 0)")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.2)

axes[0].legend()

fig.text(
    0.5,
    -0.015,
    "Lines: pooled samples at T=0.5, 1.0, 1.5. "
    "X markers at T=0 are greedy references using the size-matched "
    "sample0 scope.",
    ha="center",
    fontsize=9
)

fig.tight_layout()
fig.savefig(
    DIRS["plots"] / "temperature_effects.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close(fig)

print(
    metrics_df[
        (metrics_df.group == "main")
        & (metrics_df.scope == "pooled")
    ][
        [
            "model", "condition", "distinct1", "distinct2",
            "rep3", "loop_rate", "bigram_valid", "trigram_valid",
            "seed_carryover", "quote_closure"
        ]
    ].to_string(index=False)
)

     model condition  distinct1  distinct2     rep3  loop_rate  bigram_valid  trigram_valid  seed_carryover  quote_closure
simple_rnn      T0.5   0.186667   0.476190 0.039583   0.500000      0.978231       0.870833        0.098333       0.400000
simple_rnn      T1.0   0.303333   0.778912 0.015278   0.200000      0.846259       0.488194        0.025000       0.533333
simple_rnn      T1.5   0.490000   0.968707 0.000000   0.000000      0.535374       0.108333        0.023333       0.666667
simple_rnn    greedy   0.116667   0.187075 0.500000   1.000000      1.000000       0.968750        0.083333       0.833333
     birnn      T0.5   0.202667   0.508844 0.041667   0.433333      0.965306       0.832639        0.095000       0.366667
     birnn      T1.0   0.299333   0.763946 0.011806   0.166667      0.879592       0.513889        0.058333       0.333333
     birnn      T1.5   0.430000   0.943537 0.000000   0.000000      0.619728       0.150000        0.061667       0.700000
     birnn    gr

## Section 15 — Blind manual rating kit (48 samples)

In [17]:
def build_rating_kit(generation_rows,shuffle_seed=4242):
    selected=[r for r in generation_rows if r["group"]=="main" and r["seed_id"] in {"S2","S4"}
              and r["sample_index"] in {0,1} and r["condition"] in {"T0.5","T1.0","T1.5"}]
    assert len(selected)==48, f"Expected 48 rating items, found {len(selected)}."
    rng=random.Random(shuffle_seed); rng.shuffle(selected)
    template=[]; key=[]
    for i,r in enumerate(selected,start=1):
        template.append({"item_id":i,"text":r["seed_text"]+" [GENERATED] "+r["text"],
                         "coherence":"","contextual_consistency":"","repetition_flag":"","notes":""})
        key.append({"item_id":i,"model":r["model"],"temperature":r["temperature"],"seed_id":r["seed_id"],"sample_index":r["sample_index"]})
    return pd.DataFrame(template),pd.DataFrame(key)

rating_template,key_df=build_rating_kit(rows,CFG.run.rating_shuffle_seed)
assert len(rating_template)==48 and len(key_df)==48
assert "model" not in rating_template.columns
assert set(rating_template.item_id)==set(range(1,49))==set(key_df.item_id)
write_csv(rating_template,DIRS["results"] / "manual_ratings_template.csv")
write_csv(key_df,DIRS["results"] / "blind_key.csv")
print("Created 48-item blind rating kit.")
print("Rubric (rate before opening blind_key.csv):")
print("Coherence / contextual consistency: 1=very poor, 2=weak, 3=mixed, 4=mostly good, 5=strong throughout.")
print("Repetition flag: 0=none, 1=noticeable, 2=severe loop. One rater and 48 short texts support, but do not settle, model ranking.")

Created 48-item blind rating kit.
Rubric (rate before opening blind_key.csv):
Coherence / contextual consistency: 1=very poor, 2=weak, 3=mixed, 4=mostly good, 5=strong throughout.
Repetition flag: 0=none, 1=noticeable, 2=severe loop. One rater and 48 short texts support, but do not settle, model ranking.


## Section 16 — Summary, artifact inventory and final run configuration

In [19]:
# Candidate set for the predetermined two-stage model-selection rule.
# This is a predictive shortlist only; final selection needs blinded ratings from manual_ratings_template.csv.
seed42_main=metrics_summary[(metrics_summary.group=="main")&(metrics_summary.seed==42)].copy()
if len(seed42_main)==4:
    test_rank=list(seed42_main.sort_values("test_ppl")["model"])
    val_rank=list(seed42_main.sort_values("val_ppl")["model"])
    best_test=float(seed42_main["test_ppl"].min())
    candidates=set(seed42_main.loc[seed42_main["test_ppl"]<=1.03*best_test,"model"])
    rank_disagrees=(test_rank[0]!=val_rank[0])
    if rank_disagrees:
        candidates.update([test_rank[0],val_rank[0]])
    preliminary={"test_perplexity_rank":test_rank,"validation_perplexity_rank":val_rank,
                 "top_model_disagreement":rank_disagrees,"candidate_set_for_manual_generation_stage":sorted(candidates),
                 "warning":"No final winner is selected here. Apply PLAN.md Section 4.7 after completing manual ratings."}
    write_json(preliminary,DIRS["results"] / "predictive_candidate_set.json")
    print("Test PPL ranking:",test_rank,"| Validation PPL ranking:",val_rank,"| Candidate set:",sorted(candidates))

# Write an attribution/access log based on the mounted files; verify the live listing/licence manually at submission.
source_lines=[
    "# Dataset access and provenance", "",
    "Dataset selected in the approved plan: Sherlock Holmes Complete Collection.",
    "Kaggle source: https://www.kaggle.com/datasets/bengiefigueroa/sherlock-holmes-complete-collection",
    "The approved plan records the Kaggle listing as CC0/Public Domain; verify the current license field and attribution requirements on the live page before submission or redistribution.",
    "The notebook did not download data or use network access. It read the attached input files listed in `results/dataset_file_manifest.csv`.", "",
    f"Selected raw text files: {len(txt_files)}", f"Unique non-empty works used after exact duplicate removal: {len(works)}",
    f"Total bytes across selected .txt files: {int(file_manifest_df['bytes'].sum())}", "",
    "Preprocessing: optional Project Gutenberg boilerplate removal; Unicode NFKD and combining-mark removal; curly quote and dash normalization; hard-wrapped lines joined inside paragraphs; paragraph boundaries retained; lowercase; exact normalized duplicates dropped.",
    "The raw dataset files are not copied to the output folder."
]
(DIRS["data"] / "README.md").write_text("\n".join(source_lines)+"\n",encoding="utf-8")

# Final config captures the actual measured run, including adjustments and warnings.
RUN.update({
    "final_config":asdict(CFG), "shared_digest":shared_digest(CFG), "gpu_name":gpu_name,
    "torch_version":str(torch.__version__), "numpy_version":np.__version__, "pandas_version":pd.__version__,
    "vocab_size":vocab.V,"vocab_sha256":vocab.sha256,
    "N_by_split_full":{s:int(len(data.positions_full[s])) for s in data.positions_full},
    "N_by_split_used":{s:int(len(data.positions[s])) for s in data.positions},
    "split_seed":RUN.get("split_seed"), "seeds_run":RUN.get("seeds_run",[]),
    "optional_ablation_trained":RUN.get("optional_ablation_trained",False),
    "extra_seeds_decision":RUN.get("extra_seeds_decision"),
    "total_elapsed_hours":(time.time()-RUN["started_unix"])/3600,
    "generation_rows":len(rows), "generation_main_expected":expected_main,
    "manual_rating_items":len(rating_template), "global_flags":global_flags_snapshot(),
    "peak_gpu_memory_bytes":int(torch.cuda.max_memory_allocated(device)) if torch.cuda.is_available() else None,
})

# Expected files from the plan; list missing outputs explicitly instead of hiding omissions.
expected_output_paths=[
    DIRS["results"] / "environment.txt",DIRS["results"] / "run_config.json",DIRS["results"] / "corpus_stats.json",
    DIRS["results"] / "vocab.json",DIRS["results"] / "split_manifest.csv",DIRS["results"] / "leakage_check.json",
    DIRS["results"] / "metrics_summary.csv",DIRS["results"] / "metrics_summary.md",DIRS["results"] / "baselines.csv",
    DIRS["results"] / "seed_check.csv",DIRS["results"] / "generation_metrics.csv",
    DIRS["results"] / "manual_ratings_template.csv",DIRS["results"] / "blind_key.csv",
    DIRS["plots"] / "corpus_eda.png",DIRS["plots"] / "loss_curves.png",DIRS["plots"] / "val_perplexity_curves.png",
    DIRS["plots"] / "test_metrics_bars.png",DIRS["plots"] / "temperature_effects.png",
    DIRS["samples"] / "generation_samples.jsonl",DIRS["samples"] / "generation_samples.csv",DIRS["samples"] / "generation_samples.md",
    DIRS["data"] / "README.md",
]
missing=[str(p) for p in expected_output_paths if not p.exists()]
if missing: RUN["warnings"].append("Missing expected output files: "+", ".join(missing))

inventory=[]
for p in sorted(OUT.rglob("*")):
    if p.is_file(): inventory.append({"path":str(p.relative_to(OUT)),"bytes":p.stat().st_size})
write_csv(pd.DataFrame(inventory),DIRS["results"] / "output_inventory.csv")
file_count=len(inventory)
total_bytes=sum(x["bytes"] for x in inventory)
RUN["output_file_count_before_final_config"]=file_count
RUN["output_total_bytes_before_final_config"]=total_bytes
RUN["missing_expected_output_files"]=missing
RUN["total_elapsed_hours"]=(time.time()-RUN["started_unix"])/3600
write_json(RUN,DIRS["results"] / "run_config.json")

print("\nFinal model metrics:")
print(metrics_summary[(metrics_summary.group=="main")&(metrics_summary.seed==42)][["model","params","val_ppl","test_ppl","test_top1","test_top1_words_only","train_time_s","best_epoch"]].to_string(index=False))
print("\nGeneration metrics written:",DIRS["results"] / "generation_metrics.csv")
print("Manual ratings template:",DIRS["results"] / "manual_ratings_template.csv")
print("Output inventory:",file_count,"files,",round(total_bytes/1024/1024,2),"MiB (inventory measured before final run_config write).")
print("Missing expected artifacts:",missing if missing else "none")
print("Total elapsed hours:",round(RUN["total_elapsed_hours"],3),"(target < 6h)")
print("Warnings:")
for w in RUN["warnings"]: print(" -",w)
if missing: raise RuntimeError("Some required artifacts were not created; inspect missing_expected_output_files above.")
if file_count >= 500: raise RuntimeError(f"Output inventory has {file_count} files; plan limit is <500.")

Test PPL ranking: ['lstm', 'gru', 'simple_rnn', 'birnn'] | Validation PPL ranking: ['lstm', 'gru', 'simple_rnn', 'birnn'] | Candidate set: ['lstm']

Final model metrics:
     model   params   val_ppl   test_ppl  test_top1  test_top1_words_only  train_time_s  best_epoch
simple_rnn  7532952 72.378717  79.135110   0.239089              0.161535   1285.993919          20
     birnn 13379992 95.140369 104.892233   0.235255              0.157637   2962.580282          20
      lstm 10291608 57.382926  63.540011   0.262265              0.174436   1744.486036           6
       gru  9372056 62.340702  68.622219   0.250618              0.172756   1386.409853           7

Generation metrics written: /kaggle/working/results/generation_metrics.csv
Manual ratings template: /kaggle/working/results/manual_ratings_template.csv
Output inventory: 45 files, 779.71 MiB (inventory measured before final run_config write).
Missing expected artifacts: none
Total elapsed hours: 2.894 (target < 6h)
Warnings:


In [25]:
from pathlib import Path
from IPython.display import FileLink, display
import zipfile
import json
import os
from datetime import datetime

# ============================================================
# 1. Paths
# ============================================================
WORKING = Path("/kaggle/working")
ZIP_PATH = WORKING / "fgai_ia2_github_bundle_corrected.zip"
MANIFEST_PATH = WORKING / "bundle_manifest_corrected.json"

assert WORKING.exists(), f"Working directory not found: {WORKING}"

# ============================================================
# 2. Include only useful assignment/repository file types
# ============================================================
ALLOWED_SUFFIXES = {
    ".ipynb", ".py",
    ".md", ".txt", ".log",
    ".csv", ".json", ".jsonl",
    ".yaml", ".yml", ".toml", ".ini",
    ".png", ".jpg", ".jpeg", ".webp", ".svg",
    ".pdf", ".html",
}

EXCLUDED_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "checkpoints",
    "model_checkpoints",
    "weights",
    "wandb",
    "datasets",
    "dataset",
    "raw_data",
    "raw_dataset",
    "text_corpus",
}

EXCLUDED_FILES = {
    "blind_key.csv",
    "bundle_manifest.json",
    ZIP_PATH.name,
    "fgai_ia2_github_bundle.zip",
}

EXCLUDED_SUFFIXES = {
    ".zip", ".pt", ".pth", ".onnx",
    ".h5", ".keras", ".ckpt",
    ".safetensors", ".pkl", ".pickle",
    ".npy", ".npz",
}

MAX_FILE_SIZE_MB = 100

# ============================================================
# 3. Find relevant files throughout /kaggle/working
# ============================================================
files_to_zip = []
skipped_large = []

for root, dirs, files in os.walk(WORKING):
    root_path = Path(root)

    # Prevent walking into excluded directories
    dirs[:] = [
        d for d in dirs
        if d.lower() not in EXCLUDED_DIRS
    ]

    for filename in files:
        path = root_path / filename

        if not path.is_file():
            continue

        # Never include any ZIP archive, including the old bundle
        if path.suffix.lower() in EXCLUDED_SUFFIXES:
            continue

        if filename in EXCLUDED_FILES:
            continue

        # Exclude raw Sherlock text files if stored in working
        if (
            "sherlock" in filename.lower()
            and path.suffix.lower() in {".txt", ".epub"}
        ):
            continue

        # Keep the bundle reasonably sized
        size_mb = path.stat().st_size / (1024 ** 2)
        if size_mb > MAX_FILE_SIZE_MB:
            skipped_large.append((str(path), round(size_mb, 2)))
            continue

        # Include supported repository/result files
        if path.suffix.lower() in ALLOWED_SUFFIXES:
            files_to_zip.append(path)

# Avoid duplicate paths
files_to_zip = sorted(set(files_to_zip))

# ============================================================
# 4. Diagnose whether plots, samples, and notebooks exist
# ============================================================
def relative_name(path):
    return str(path.relative_to(WORKING))

plot_files = [
    p for p in files_to_zip
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".svg"}
    or any(part.lower() in {"plots", "plot", "figures", "figure"}
           for part in p.relative_to(WORKING).parts[:-1])
]

sample_files = [
    p for p in files_to_zip
    if any(
        word in relative_name(p).lower()
        for word in [
            "samples/",
            "generated_text",
            "generation_samples",
            "generated_samples",
            "text_samples",
        ]
    )
    and "metrics" not in p.name.lower()
]

notebook_files = [
    p for p in files_to_zip
    if p.suffix.lower() == ".ipynb"
]

print("Files discovered:", len(files_to_zip))
print("Plot/figure files:", len(plot_files))
print("Generated-sample candidate files:", len(sample_files))
print("Notebook files:", len(notebook_files))

print("\nPlot files:")
for p in plot_files:
    print(" -", relative_name(p))

print("\nSample-related files:")
for p in sample_files:
    print(" -", relative_name(p))

print("\nNotebook files:")
for p in notebook_files:
    print(" -", relative_name(p))

# ============================================================
# 5. Write a manifest for reproducibility
# ============================================================
manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_directory": str(WORKING),
    "file_count_excluding_manifest": len(files_to_zip),
    "plots_found": len(plot_files),
    "sample_related_files_found": len(sample_files),
    "notebooks_found": len(notebook_files),
    "excluded_private_file": "blind_key.csv",
    "files": [relative_name(p) for p in files_to_zip],
    "large_files_skipped": skipped_large,
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

# ============================================================
# 6. Create corrected ZIP at the top level of working
# ============================================================
with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:
    for path in files_to_zip:
        zf.write(
            path,
            arcname=relative_name(path)
        )

    zf.write(
        MANIFEST_PATH,
        arcname="bundle_manifest_corrected.json"
    )

# ============================================================
# 7. Verify and provide download link
# ============================================================
assert ZIP_PATH.exists(), "ZIP was not created."
assert zipfile.is_zipfile(ZIP_PATH), "ZIP validation failed."

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    if bad_file is not None:
        raise RuntimeError(f"Corrupt file in ZIP: {bad_file}")

print("\n" + "=" * 55)
print("CORRECTED ZIP CREATED AND VERIFIED")
print("Location:", ZIP_PATH)
print("Size:", round(ZIP_PATH.stat().st_size / (1024 ** 2), 2), "MB")
print("Archive integrity: OK")
print("=" * 55)

if not plot_files:
    print(
        "\nWARNING: No saved plot images were found. "
        "Re-run the plotting cells before creating the final bundle."
    )

if not sample_files:
    print(
        "\nWARNING: No obvious generated-text sample files were found. "
        "Re-run the text-generation/export cells before creating the final bundle."
    )

if not notebook_files:
    print(
        "\nNOTE: No notebook file was found under /kaggle/working. "
        "Add your .ipynb file to working if you want it inside this ZIP."
    )

if skipped_large:
    print("\nLarge files skipped:")
    for filename, size in skipped_large:
        print(f" - {filename} ({size} MB)")

# Use a relative download link from the working directory
os.chdir(WORKING)
display(FileLink(ZIP_PATH.name))

Files discovered: 69
Plot/figure files: 10
Generated-sample candidate files: 6
Notebook files: 1

Plot files:
 - fgai_ia2_github_bundle/plots/corpus_eda.png
 - fgai_ia2_github_bundle/plots/loss_curves.png
 - fgai_ia2_github_bundle/plots/temperature_effects.png
 - fgai_ia2_github_bundle/plots/test_metrics_bars.png
 - fgai_ia2_github_bundle/plots/val_perplexity_curves.png
 - plots/corpus_eda.png
 - plots/loss_curves.png
 - plots/temperature_effects.png
 - plots/test_metrics_bars.png
 - plots/val_perplexity_curves.png

Sample-related files:
 - fgai_ia2_github_bundle/samples/generation_samples.csv
 - fgai_ia2_github_bundle/samples/generation_samples.jsonl
 - fgai_ia2_github_bundle/samples/generation_samples.md
 - samples/generation_samples.csv
 - samples/generation_samples.jsonl
 - samples/generation_samples.md

Notebook files:
 - .virtual_documents/__notebook_source__.ipynb

CORRECTED ZIP CREATED AND VERIFIED
Location: /kaggle/working/fgai_ia2_github_bundle_corrected.zip
Size: 1.66 MB
Arc

/kaggle/working/fgai_ia2_github_bundle_corrected.zip